In [9]:
import pandas as pd

tickets = pd.read_csv('../data/tickets.csv')

print(tickets.shape)
print(tickets.head())
print(tickets.columns.tolist())


(12528, 21)
   ticket_id        created_at first_response_at       resolved_at    status  \
0  TK-240001  2025-01-01 09:48  2025-01-01 11:34  2025-01-01 11:49  resolved   
1  TK-240001  2025-01-01 09:48  2025-01-01 11:34  2025-01-01 06:19  resolved   
2  TK-240002  2025-01-01 13:24  2025-01-01 13:26  2025-01-01 08:28  resolved   
3  TK-240003  2025-01-01 13:55  2025-01-01 14:06  2025-01-01 09:07  resolved   
4  TK-240004  2025-01-01 15:13  2025-01-01 17:14  2025-01-02 11:57  resolved   

  channel customer_id  order_id product_sku             category  ...  \
0   email     C106340       NaN   VA-EB-AIR      Account & Login  ...   
1   email     C106340       NaN   VA-EB-AIR      Account & Login  ...   
2    chat     C102868       NaN   VA-HP-ST2        Audio Quality  ...   
3    chat     C108259  VR894659   VA-SW-FIT   Billing & Payments  ...   
4   email     C100979  VR909673  VA-AC-CH65  Delivery & Shipping  ...   

     assigned_team agent_id transfers  csat_score  refund_amount_inr

In [2]:
tickets["ticket_id"].duplicated().sum()

653

In [3]:
tickets["ticket_id"].nunique()

11875

In [4]:
tickets[tickets["ticket_id"] == "TK-240001"][
    ["ticket_id", "source_system", "created_at", "resolved_at", "csat_score"]
]

,ticket_id,source_system,created_at,resolved_at,csat_score
0,TK-240001,helpdesk,2025-01-01 09:48,2025-01-01 11:49,5.0
1,TK-240001,legacy_fd,2025-01-01 09:48,2025-01-01 06:19,5.0


In [5]:
duplicates = tickets[tickets["ticket_id"].duplicated(keep=False)].copy()

duplicates["created_at"] = pd.to_datetime(duplicates["created_at"])
duplicates["resolved_at"] = pd.to_datetime(duplicates["resolved_at"])

pivot = (
    duplicates
    .pivot(index="ticket_id", columns="source_system", values="resolved_at")
    .reset_index()
)

pivot["difference_hours"] = (
    pivot["helpdesk"] - pivot["legacy_fd"]
).dt.total_seconds() / 3600

pivot["difference_hours"].describe()

count    618.0
mean       5.5
std        0.0
min        5.5
25%        5.5
50%        5.5
75%        5.5
max        5.5
Name: difference_hours, dtype: float64

In [6]:
duplicates = tickets[tickets["ticket_id"].duplicated(keep=False)]

print(
    pd.crosstab(
        duplicates["ticket_id"],
        duplicates["source_system"]
    ).value_counts()
)

helpdesk  legacy_fd
1         1            653
Name: count, dtype: int64


In [7]:
duplicates.groupby("source_system")["ticket_id"].nunique()

source_system
helpdesk     653
legacy_fd    653
Name: ticket_id, dtype: int64

In [8]:
tickets["source_priority"] = tickets["source_system"].map({
    "helpdesk": 1,
    "legacy_fd": 2
})
tickets = tickets.sort_values(
    ["ticket_id", "source_priority"]
)

clean_tickets = tickets.drop_duplicates(
    subset="ticket_id",
    keep="first"
)
clean_tickets = clean_tickets.drop(
    columns="source_priority"
)
print("Raw rows:", len(tickets))
print("Clean rows:", len(clean_tickets))
print("Unique tickets:", clean_tickets["ticket_id"].nunique())
print("Duplicate IDs:", clean_tickets["ticket_id"].duplicated().sum())

Raw rows: 12528
Clean rows: 11875
Unique tickets: 11875
Duplicate IDs: 0


In [9]:
date_cols = [
    "created_at",
    "first_response_at",
    "resolved_at"
]

for col in date_cols:
    clean_tickets[col] = pd.to_datetime(
        clean_tickets[col],
        errors="coerce"
    )

In [10]:
clean_tickets[date_cols].dtypes

created_at           datetime64[ns]
first_response_at    datetime64[ns]
resolved_at          datetime64[ns]
dtype: object

In [11]:
clean_tickets[date_cols].isna().sum()

created_at             0
first_response_at      0
resolved_at          609
dtype: int64

In [12]:
bad_response = clean_tickets[
    clean_tickets["first_response_at"] < clean_tickets["created_at"]
]

bad_resolution = clean_tickets[
    clean_tickets["resolved_at"] < clean_tickets["created_at"]
]

print("Response before creation:", len(bad_response))
print("Resolution before creation:", len(bad_resolution))

Response before creation: 0
Resolution before creation: 1874


In [13]:
print(
    clean_tickets.loc[
        clean_tickets["resolved_at"] < clean_tickets["created_at"],
        "source_system"
    ].value_counts()
)

source_system
legacy_fd    1874
Name: count, dtype: int64


In [14]:
print(
    clean_tickets.loc[
        clean_tickets["resolved_at"] < clean_tickets["created_at"],
        ["ticket_id", "source_system", "created_at",
         "first_response_at", "resolved_at", "status"]
    ].head(10)
)

    ticket_id source_system          created_at   first_response_at  \
2   TK-240002     legacy_fd 2025-01-01 13:24:00 2025-01-01 13:26:00   
3   TK-240003     legacy_fd 2025-01-01 13:55:00 2025-01-01 14:06:00   
7   TK-240006     legacy_fd 2025-01-01 21:01:00 2025-01-01 22:51:00   
8   TK-240007     legacy_fd 2025-01-01 23:31:00 2025-01-02 02:01:00   
11  TK-240010     legacy_fd 2025-01-02 16:14:00 2025-01-02 17:32:00   
12  TK-240011     legacy_fd 2025-01-02 19:18:00 2025-01-02 19:20:00   
13  TK-240012     legacy_fd 2025-01-02 19:53:00 2025-01-02 20:08:00   
18  TK-240016     legacy_fd 2025-01-02 21:39:00 2025-01-02 22:11:00   
21  TK-240020     legacy_fd 2025-01-03 07:07:00 2025-01-03 07:13:00   
25  TK-240025     legacy_fd 2025-01-03 10:22:00 2025-01-03 10:25:00   

           resolved_at    status  
2  2025-01-01 08:28:00  resolved  
3  2025-01-01 09:07:00  resolved  
7  2025-01-01 17:43:00  resolved  
8  2025-01-01 21:08:00  resolved  
11 2025-01-02 12:17:00    closed  
12 2025-

In [15]:
clean_tickets["resolved_at_normalized"] = clean_tickets["resolved_at"]

legacy_mask = clean_tickets["source_system"] == "legacy_fd"

clean_tickets.loc[legacy_mask, "resolved_at_normalized"] = (
    clean_tickets.loc[legacy_mask, "resolved_at"]
    + pd.Timedelta(hours=5, minutes=30)
)

In [16]:
bad_resolution = clean_tickets[
    clean_tickets["resolved_at_normalized"] < clean_tickets["created_at"]
]

print("Resolution before creation:", len(bad_resolution))

Resolution before creation: 0


In [17]:
print(
    clean_tickets[
        ["source_system", "resolved_at", "resolved_at_normalized"]
    ].head(10)
)

   source_system         resolved_at resolved_at_normalized
0       helpdesk 2025-01-01 11:49:00    2025-01-01 11:49:00
2      legacy_fd 2025-01-01 08:28:00    2025-01-01 13:58:00
3      legacy_fd 2025-01-01 09:07:00    2025-01-01 14:37:00
4      legacy_fd 2025-01-02 11:57:00    2025-01-02 17:27:00
5       helpdesk 2025-01-03 20:52:00    2025-01-03 20:52:00
7      legacy_fd 2025-01-01 17:43:00    2025-01-01 23:13:00
8      legacy_fd 2025-01-01 21:08:00    2025-01-02 02:38:00
9      legacy_fd 2025-01-07 03:08:00    2025-01-07 08:38:00
10     legacy_fd 2025-01-07 05:58:00    2025-01-07 11:28:00
11     legacy_fd 2025-01-02 12:17:00    2025-01-02 17:47:00


In [18]:
clean_tickets["handle_time"] = (
    clean_tickets["resolved_at_normalized"]
    - clean_tickets["first_response_at"]
)

print(clean_tickets["handle_time"].describe())

count                        11266
mean     0 days 22:57:14.086632345
std      2 days 01:27:24.586096872
min                0 days 00:05:00
25%                0 days 00:18:00
50%                0 days 00:27:00
75%                1 days 00:24:00
max               16 days 02:06:00
Name: handle_time, dtype: object


In [19]:
missing_resolution = clean_tickets[
    clean_tickets["resolved_at_normalized"].isna()
]

print(
    missing_resolution["status"].value_counts()
)

status
open       376
pending    233
Name: count, dtype: int64


In [20]:
print(
    pd.crosstab(
        clean_tickets["status"],
        clean_tickets["resolved_at_normalized"].isna()
    )
)

resolved_at_normalized  False  True 
status                              
closed                   1107      0
open                        0    376
pending                     0    233
resolved                10159      0


In [21]:
print(clean_tickets["csat_score"].value_counts(dropna=False).sort_index())

csat_score
0.0    1750
1.0     267
2.0     894
3.0    1737
4.0    1653
5.0     718
NaN    4856
Name: count, dtype: int64


In [22]:
print(
    "Invalid CSAT:",
    ((clean_tickets["csat_score"] < 1) |
     (clean_tickets["csat_score"] > 5)).sum()
)

Invalid CSAT: 1750


In [23]:
clean_tickets["csat_score"] = clean_tickets["csat_score"].replace(0, pd.NA)

print(clean_tickets["csat_score"].value_counts(dropna=False).sort_index())

csat_score
1.0      267
2.0      894
3.0     1737
4.0     1653
5.0      718
NaN     4856
<NA>    1750
Name: count, dtype: int64


In [24]:
print("Valid CSAT responses:",
      clean_tickets["csat_score"].notna().sum())

Valid CSAT responses: 5269


In [25]:
clean_tickets["csat_score"] = (
    clean_tickets["csat_score"].replace(0, pd.NA)
)

In [26]:
print(
    clean_tickets["csat_score"]
    .value_counts(dropna=False)
    .sort_index()
)

csat_score
1.0      267
2.0      894
3.0     1737
4.0     1653
5.0      718
NaN     4856
<NA>    1750
Name: count, dtype: int64


In [27]:
for col in ["status", "channel", "priority", "category", "assigned_team", "source_system"]:
    print(f"\n--- {col} ---")
    print(clean_tickets[col].value_counts(dropna=False))


--- status ---
status
resolved    10159
closed       1107
open          376
pending       233
Name: count, dtype: int64

--- channel ---
channel
chat      5161
email     3807
voice     1704
social    1203
Name: count, dtype: int64

--- priority ---
priority
Normal    8479
High      2065
Low       1331
Name: count, dtype: int64

--- category ---
category
Delivery & Shipping    2134
Other                  1691
Billing & Payments     1624
Returns & Refunds      1197
Connectivity           1131
Charging & Battery      955
App & Firmware          822
Audio Quality           792
Warranty & Repair       620
Product Enquiry         593
Account & Login         316
Name: count, dtype: int64

--- assigned_team ---
assigned_team
Chat Frontline            3399
Logistics                 2134
Email Frontline           1987
Billing                   1624
Returns Desk              1197
Voice Frontline            914
Escalations & Warranty     620
Name: count, dtype: int64

--- source_system ---
source

In [28]:
print(clean_tickets["transfers"].describe())
print("Negative transfers:",
      (clean_tickets["transfers"] < 0).sum())
print("Missing transfers:",
      clean_tickets["transfers"].isna().sum())

count    11875.000000
mean         0.098442
std          0.332396
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          2.000000
Name: transfers, dtype: float64
Negative transfers: 0
Missing transfers: 0


## Validate transfers by source

In [29]:
clean_tickets.groupby("source_system")["transfers"].agg(
    ["count", "sum", "mean", "max"]
)

,count,sum,mean,max
source_system,,,,
helpdesk,8766,868,0.099019,2
legacy_fd,3109,301,0.096816,2


## Check refund data

In [30]:
clean_tickets["refund_amount_inr"].describe()

count     2105.000000
mean      2846.992399
std       2258.847450
min         57.000000
25%       1169.000000
50%       2499.000000
75%       3499.000000
max      13998.000000
Name: refund_amount_inr, dtype: float64

In [31]:
print(
    "Refund records:",
    clean_tickets["refund_amount_inr"].notna().sum()
)

print(
    "Negative refunds:",
    (clean_tickets["refund_amount_inr"] < 0).sum()
)

Refund records: 2105
Negative refunds: 0


In [32]:
clean_tickets.groupby("source_system")[
    "refund_amount_inr"
].agg(["count", "sum", "mean", "min", "max"])

,count,sum,mean,min,max
source_system,,,,,
helpdesk,1524,4225922.0,2772.914698,60.0,13998.0
legacy_fd,581,1766997.0,3041.302926,57.0,13998.0


## Check replacement values

In [33]:
print(clean_tickets["replacement_issued"].value_counts(dropna=False))

replacement_issued
N    10673
Y     1202
Name: count, dtype: int64


In [34]:
pd.crosstab(
    clean_tickets["replacement_issued"],
    clean_tickets["refund_reason_code"],
    dropna=False
)

refund_reason_code,CANCEL,DOA-REPL,DUP-PAYMENT,GW-OTHER,LOST-TRANSIT,PRICE-ADJ,RETURN-QC-OK,WTY-BUYBACK,NaN
replacement_issued,,,,,,,,,
N,296,135,531,41,91,120,835,52,8572
Y,0,1,0,1,0,0,2,0,1198


In [35]:
clean_tickets[
    (clean_tickets["source_system"] == "legacy_fd") &
    (clean_tickets["transfers"] > 0)
][[
    "ticket_id",
    "source_system",
    "created_at",
    "transfers",
    "assigned_team",
    "agent_id"
]].head(20)

,ticket_id,source_system,created_at,transfers,assigned_team,agent_id
7,TK-240006,legacy_fd,2025-01-01 21:01:00,1,Returns Desk,A3036
13,TK-240012,legacy_fd,2025-01-02 19:53:00,1,Chat Frontline,A3008
56,TK-240055,legacy_fd,2025-01-05 16:14:00,1,Returns Desk,A3028
60,TK-240059,legacy_fd,2025-01-06 09:24:00,1,Email Frontline,A3028
61,TK-240061,legacy_fd,2025-01-06 09:28:00,1,Logistics,A3008
67,TK-240068,legacy_fd,2025-01-06 16:34:00,1,Escalations & Warranty,A3038
74,TK-240074,legacy_fd,2025-01-07 10:44:00,1,Chat Frontline,A3040
83,TK-240088,legacy_fd,2025-01-08 10:51:00,1,Chat Frontline,A3030
91,TK-240099,legacy_fd,2025-01-09 18:02:00,1,Email Frontline,A3010
108,TK-240118,legacy_fd,2025-01-11 10:40:00,1,Billing,A3037


In [36]:
clean_tickets[
    (clean_tickets["source_system"] == "legacy_fd") &
    (clean_tickets["transfers"] > 0)
]["transfers"].value_counts()

transfers
1    235
2     33
Name: count, dtype: int64

In [37]:
replacement_refund = clean_tickets[
    (clean_tickets["replacement_issued"] == "Y") &
    (clean_tickets["refund_amount_inr"].notna())
]

print("Replacement + actual refund:", len(replacement_refund))

print(
    replacement_refund[
        [
            "ticket_id",
            "order_id",
            "refund_amount_inr",
            "refund_reason_code",
            "replacement_issued"
        ]
    ].head(20)
)

Replacement + actual refund: 4
       ticket_id  order_id  refund_amount_inr refund_reason_code  \
1291   TK-241405  VR892778             2499.0       RETURN-QC-OK   
4072   TK-244372  VR908861             2974.0           GW-OTHER   
9020   TK-250483  VR883328             2878.0           DOA-REPL   
10558  TK-252411  VR885453             3499.0       RETURN-QC-OK   

      replacement_issued  
1291                   Y  
4072                   Y  
9020                   Y  
10558                  Y  


In [38]:
replacement_refund[
    [
        "ticket_id",
        "customer_id",
        "order_id",
        "product_sku",
        "category",
        "refund_amount_inr",
        "refund_reason_code",
        "replacement_issued",
        "customer_message",
        "agent_notes"
    ]
]


,ticket_id,customer_id,order_id,product_sku,category,refund_amount_inr,refund_reason_code,replacement_issued,customer_message,agent_notes
1291,TK-241405,C105023,VR892778,VA-SP-MINI,Delivery & Shipping,2499.0,RETURN-QC-OK,Y,I ordered the mini speaker and got something e...,cx says wrong item delivered. raised with ware...
4072,TK-244372,C105537,VR908861,VA-EB-PL2,Charging & Battery,2974.0,GW-OTHER,Y,Not acceptabble at tihs price. GOT PULSE 2 BDS...,1. Left earbud not taking charge\n2. Checked c...
9020,TK-250483,C106059,VR883328,VA-EB-AIR,Delivery & Shipping,2878.0,DOA-REPL,Y,"[IVR transcript] Namaste, airlite earbuds purc...",Cust contacted re transit damage. Checkked wit...
10558,TK-252411,C104944,VR885453,VA-EB-PL2,Delivery & Shipping,3499.0,RETURN-QC-OK,Y,[IVR transcript] HELLO JI GOT A DIFFERENT COLO...,1. Incorrect product shipped\n2. Raised w/ war...


In [39]:
clean_tickets.groupby("source_system")["refund_amount_inr"].agg(
    ["count", "sum", "mean", "min", "max"]
)

,count,sum,mean,min,max
source_system,,,,,
helpdesk,1524,4225922.0,2772.914698,60.0,13998.0
legacy_fd,581,1766997.0,3041.302926,57.0,13998.0


In [40]:
clean_tickets["refund_replacement_conflict"] = (
    clean_tickets["replacement_issued"].eq("Y")
    & clean_tickets["refund_amount_inr"].notna()
)

In [41]:
clean_tickets["refund_replacement_conflict"].value_counts()

refund_replacement_conflict
False    11871
True         4
Name: count, dtype: int64

In [42]:
replacement_refund[
    [
        "ticket_id",
        "source_system",
        "order_id",
        "refund_amount_inr",
        "refund_reason_code",
        "replacement_issued"
    ]
]

,ticket_id,source_system,order_id,refund_amount_inr,refund_reason_code,replacement_issued
1291,TK-241405,legacy_fd,VR892778,2499.0,RETURN-QC-OK,Y
4072,TK-244372,legacy_fd,VR908861,2974.0,GW-OTHER,Y
9020,TK-250483,helpdesk,VR883328,2878.0,DOA-REPL,Y
10558,TK-252411,helpdesk,VR885453,3499.0,RETURN-QC-OK,Y


In [43]:
final_tickets = clean_tickets.copy()

In [44]:
final_tickets.to_csv(
    "../data/clean_tickets.csv",
    index=False
)

In [45]:
print(final_tickets.shape)
print(final_tickets["ticket_id"].nunique())
print(final_tickets["ticket_id"].duplicated().sum())

(11875, 24)
11875
0


## KPI #1: SLA breaches

In [46]:
final_tickets["first_response_delay"] = (
    final_tickets["first_response_at"]
    - final_tickets["created_at"]
)

In [47]:
final_tickets["first_response_delay"].describe()

count                        11875
mean     0 days 01:38:35.459368421
std      0 days 03:11:13.012203668
min                0 days 00:00:00
25%                0 days 00:05:00
50%                0 days 00:26:00
75%                0 days 01:53:00
max                2 days 11:36:00
Name: first_response_delay, dtype: object

In [48]:
sla_targets = {
    "chat": pd.Timedelta(minutes=15),
    "voice": pd.Timedelta(hours=2),
    "social": pd.Timedelta(hours=4),
    "email": pd.Timedelta(hours=8)
}

In [49]:
final_tickets["sla_target"] = final_tickets["channel"].map(sla_targets)

In [50]:
final_tickets[
    ["channel", "first_response_delay", "sla_target"]
].head(10)

,channel,first_response_delay,sla_target
0,email,0 days 01:46:00,0 days 08:00:00
2,chat,0 days 00:02:00,0 days 00:15:00
3,chat,0 days 00:11:00,0 days 00:15:00
4,email,0 days 02:01:00,0 days 08:00:00
5,voice,0 days 00:13:00,0 days 02:00:00
7,email,0 days 01:50:00,0 days 08:00:00
8,email,0 days 02:30:00,0 days 08:00:00
9,chat,0 days 00:06:00,0 days 00:15:00
10,email,0 days 02:01:00,0 days 08:00:00
11,voice,0 days 01:18:00,0 days 02:00:00


In [51]:
final_tickets["sla_breach"] = (
    final_tickets["first_response_delay"]
    > final_tickets["sla_target"]
)

In [52]:
print(final_tickets["sla_breach"].value_counts())

sla_breach
False    10824
True      1051
Name: count, dtype: int64


In [53]:
sla_breach_count = final_tickets["sla_breach"].sum()
total_tickets = len(final_tickets)

sla_breach_rate = sla_breach_count / total_tickets

print("SLA breaches:", sla_breach_count)
print("SLA breach rate:", sla_breach_rate)
print("SLA breach rate %:", sla_breach_rate * 100)

SLA breaches: 1051
SLA breach rate: 0.08850526315789474
SLA breach rate %: 8.850526315789473


In [54]:
sla_credit_exposure = sla_breach_count * 350

print("SLA credit exposure: ₹", sla_credit_exposure)

SLA credit exposure: ₹ 367850


In [55]:
final_tickets.groupby("channel")["sla_breach"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
channel,,,
chat,5161,424,0.082155
email,3807,440,0.115577
social,1203,91,0.075644
voice,1704,96,0.056338


In [56]:
total_transfers = final_tickets["transfers"].sum()

transfer_cost = total_transfers * 305

print("Total transfers:", total_transfers)
print("Transfer cost: ₹", transfer_cost)

Total transfers: 1169
Transfer cost: ₹ 356545


In [57]:
final_tickets.groupby("assigned_team")["transfers"].agg(
    ["count", "sum", "mean", "max"]
)

,count,sum,mean,max
assigned_team,,,,
Billing,1624,134,0.082512,2
Chat Frontline,3399,377,0.110915,2
Email Frontline,1987,235,0.118269,2
Escalations & Warranty,620,50,0.080645,2
Logistics,2134,169,0.079194,2
Returns Desk,1197,106,0.088555,2
Voice Frontline,914,98,0.107221,2


In [58]:
current_helpdesk = final_tickets[
    final_tickets["source_system"] == "helpdesk"
]

current_transfers = current_helpdesk["transfers"].sum()
current_transfer_cost = current_transfers * 305

print("Current helpdesk transfers:", current_transfers)
print("Current transfer cost: ₹", current_transfer_cost)

Current helpdesk transfers: 868
Current transfer cost: ₹ 264740


In [59]:
final_tickets[
    ["customer_id", "product_sku", "category", "created_at", "resolved_at_normalized"]
].head()

,customer_id,product_sku,category,created_at,resolved_at_normalized
0,C106340,VA-EB-AIR,Account & Login,2025-01-01 09:48:00,2025-01-01 11:49:00
2,C102868,VA-HP-ST2,Audio Quality,2025-01-01 13:24:00,2025-01-01 13:58:00
3,C108259,VA-SW-FIT,Billing & Payments,2025-01-01 13:55:00,2025-01-01 14:37:00
4,C100979,VA-AC-CH65,Delivery & Shipping,2025-01-01 15:13:00,2025-01-02 17:27:00
5,C102389,VA-EB-PL1,Delivery & Shipping,2025-01-01 20:24:00,2025-01-03 20:52:00


## Repeat contants


In [60]:
repeat_df = final_tickets.sort_values(
    ["customer_id", "created_at"]
).copy()

In [61]:
previous = repeat_df[
    [
        "ticket_id",
        "customer_id",
        "product_sku",
        "category",
        "created_at",
        "resolved_at_normalized"
    ]
].copy()

previous = previous.rename(columns={
    "ticket_id": "previous_ticket_id",
    "product_sku": "previous_product_sku",
    "category": "previous_category",
    "created_at": "previous_created_at",
    "resolved_at_normalized": "previous_resolved_at"
})

In [62]:
pairs = repeat_df.merge(
    previous,
    on="customer_id",
    how="left"
)

In [63]:
pairs["days_after_resolution"] = (
    pairs["created_at"] - pairs["previous_resolved_at"]
).dt.total_seconds() / (60 * 60 * 24)

In [64]:
repeat_candidates = pairs[
    (pairs["ticket_id"] != pairs["previous_ticket_id"]) &
    (pairs["previous_resolved_at"].notna()) &
    (pairs["days_after_resolution"] > 0) &
    (pairs["days_after_resolution"] <= 30) &
    (pairs["product_sku"] == pairs["previous_product_sku"]) &
    (pairs["category"] == pairs["previous_category"])
].copy()

In [65]:
print("Candidate repeat pairs:", len(repeat_candidates))

Candidate repeat pairs: 1527


In [66]:
print(
    repeat_candidates[
        [
            "ticket_id",
            "previous_ticket_id",
            "customer_id",
            "product_sku",
            "category",
            "days_after_resolution"
        ]
    ].head(20)
)


     ticket_id previous_ticket_id customer_id product_sku  \
6    TK-245078          TK-244806     C100001   VA-SW-NX2   
18   TK-245676          TK-244806     C100001   VA-SW-NX2   
19   TK-245676          TK-245078     C100001   VA-SW-NX2   
25   TK-245949          TK-245078     C100001   VA-SW-NX2   
27   TK-245949          TK-245676     C100001   VA-SW-NX2   
40   TK-248985          TK-248658     C100002   VA-SP-ORB   
57   TK-244284          TK-244150     C100003   VA-SW-FIT   
91   TK-245080          TK-244241     C100006   VA-EB-AIR   
95   TK-245209          TK-244241     C100006   VA-EB-AIR   
106  TK-243660          TK-243098     C100009   VA-EB-PL1   
125  TK-240812          TK-240702     C100017   VA-EB-AIR   
266  TK-243102          TK-243092     C100060   VA-EB-PL1   
280  TK-244409          TK-244127     C100062   VA-HP-ST3   
308  TK-245407          TK-245320     C100081   VA-EB-PL2   
312  TK-245443          TK-245320     C100081   VA-EB-PL2   
313  TK-245443          

In [67]:
candidate_repeat_tickets = (
    repeat_candidates["ticket_id"]
    .nunique()
)

print(
    "Unique candidate repeat tickets:",
    candidate_repeat_tickets
)

Unique candidate repeat tickets: 1414


In [68]:
repeat_frequency = (
    repeat_candidates
    .groupby("ticket_id")
    .size()
    .value_counts()
    .sort_index()
)

print(repeat_frequency)

1    1303
2     109
3       2
Name: count, dtype: int64


In [69]:
sample_pairs = repeat_candidates[
    [
        "ticket_id",
        "previous_ticket_id",
        "customer_id",
        "product_sku",
        "category",
        "days_after_resolution",
        "customer_message"
    ]
].sample(20, random_state=42)

sample_pairs

,ticket_id,previous_ticket_id,customer_id,product_sku,category,days_after_resolution,customer_message
36192,TK-246325,TK-245741,C108588,VA-SP-MINI,Charging & Battery,15.465972,Honestly regretting thiis purchase. Same issue...
1983,TK-243148,TK-242743,C100566,VA-SP-ORB,Audio Quality,23.528472,"hi team,\nordered the orbit smart speaker 2 we..."
26873,TK-252463,TK-252030,C106372,VA-SP-ORB,Delivery & Shipping,12.043750,This is the second time I am writing about thi...
17193,TK-247378,TK-246749,C104080,VA-SP-ORB,Delivery & Shipping,16.104167,"[IVR transcript] Hey, Tihs is the second time ..."
30335,TK-248321,TK-247401,C107247,VA-EB-AIR,Charging & Battery,23.259722,hi\ndies by lunchtime with light use\nhelp
26128,TK-244423,TK-243691,C106253,VA-AC-CASE,Delivery & Shipping,24.615972,helo\nfollowing up on my earlier complaint - n...
38478,TK-250867,TK-250073,C109120,VA-EB-PL2,Other,23.359722,"Namaste, I bought the Pulse 2 earbuds on 10 No..."
39510,TK-253156,TK-252778,C109314,VA-EB-AIR,Delivery & Shipping,11.429167,honestly regretting this purchase. my earlier...
39514,TK-253656,TK-252778,C109314,VA-EB-AIR,Delivery & Shipping,24.827083,yet again. - i haven't received my order - vr8...
31822,TK-250264,TK-249828,C107595,VA-EB-PL2,Audio Quality,13.606250,hi\nthird time now.\nthere is a frying sound i...


In [70]:
sample_pairs = repeat_candidates[
    [
        "ticket_id",
        "previous_ticket_id",
        "customer_id",
        "product_sku",
        "category",
        "days_after_resolution",
        "customer_message"
    ]
].merge(
    final_tickets[
        ["ticket_id", "customer_message"]
    ],
    left_on="previous_ticket_id",
    right_on="ticket_id",
    suffixes=("_new", "_previous")
)

sample_pairs[
    [
        "ticket_id_new",
        "ticket_id_previous",
        "category",
        "days_after_resolution",
        "customer_message_previous",
        "customer_message_new"
    ]
].head(20)

,ticket_id_new,ticket_id_previous,category,days_after_resolution,customer_message_previous,customer_message_new
0,TK-245078,TK-244806,Other,9.317361,Really frustrating. This is regaridng my Nexa...,"To the Vireo Customer Care Team,\n\nI am writi..."
1,TK-245676,TK-244806,Other,27.565972,Really frustrating. This is regaridng my Nexa...,Very poor quality. This is teh third time. my ...
2,TK-245676,TK-245078,Other,18.232639,"To the Vireo Customer Care Team,\n\nI am writi...",Very poor quality. This is teh third time. my ...
3,TK-245949,TK-245078,Other,25.816667,"To the Vireo Customer Care Team,\n\nI am writi...",Not acceptable at this price. Third time now. ...
4,TK-245949,TK-245676,Other,7.573611,Very poor quality. This is teh third time. my ...,Not acceptable at this price. Third time now. ...
5,TK-248985,TK-248658,Other,9.988194,"Hello Vireo, This is regarding Orbit speaker. ...",This is very disappointing. Bought my Orbit ar...
6,TK-244284,TK-244150,Returns & Refunds,3.234722,[IVR transcript] This is very disappointing. ...,helo\nreturn pickup has not happened\nhelp
7,TK-245080,TK-244241,Delivery & Shipping,25.179167,"Namaste, This is regarding AirLite. My order h...","Namaste, This is regarding the AirLite buds. O..."
8,TK-245209,TK-244241,Delivery & Shipping,29.803472,"Namaste, This is regarding AirLite. My order h...","Namaste, Yet again. Got airlite earbuds from A..."
9,TK-243660,TK-243098,Returns & Refunds,26.603472,packed the box a week ago & it's still here - ...,"Hello, Third time now. Got the Pulse buds from..."


In [71]:
repeat_unique = (
    repeat_candidates
    .sort_values("previous_resolved_at")
    .drop_duplicates("ticket_id", keep="last")
    .copy()
)

In [72]:
print("Unique repeat candidates:", len(repeat_unique))

Unique repeat candidates: 1414


In [73]:
final_tickets["repeat_contact_candidate"] = (
    final_tickets["ticket_id"].isin(
        repeat_unique["ticket_id"]
    )
)

In [74]:
print(
    final_tickets["repeat_contact_candidate"].value_counts()
)

repeat_contact_candidate
False    10461
True      1414
Name: count, dtype: int64


In [75]:
candidate_repeat_rate = (
    final_tickets["repeat_contact_candidate"].mean()
)

print("Candidate repeat rate:", candidate_repeat_rate)
print("Candidate repeat rate %:", candidate_repeat_rate * 100)

Candidate repeat rate: 0.11907368421052632
Candidate repeat rate %: 11.907368421052631


In [76]:
channel_costs = {
    "chat": 210,
    "email": 260,
    "voice": 520,
    "social": 240
}

repeat_unique["repeat_contact_cost"] = (
    repeat_unique["channel"]
    .map(channel_costs)
)

In [77]:
print(
    "Candidate repeat-contact cost: ₹",
    repeat_unique["repeat_contact_cost"].sum()
)

Candidate repeat-contact cost: ₹ 367500


In [78]:
category_repeat = (
    repeat_unique
    .groupby("category")
    .size()
    .sort_values(ascending=False)
)

print(category_repeat)

category
Delivery & Shipping    326
Billing & Payments     203
Returns & Refunds      161
Connectivity           150
Other                  147
Charging & Battery     137
Audio Quality          116
App & Firmware          99
Warranty & Repair       72
Product Enquiry          2
Account & Login          1
dtype: int64


In [79]:
category_total = (
    final_tickets
    .groupby("category")
    .size()
)

category_repeat_rate = (
    category_repeat / category_total
).sort_values(ascending=False)

print(
    category_repeat_rate.mul(100).round(2)
)

category
Delivery & Shipping    15.28
Audio Quality          14.65
Charging & Battery     14.35
Returns & Refunds      13.45
Connectivity           13.26
Billing & Payments     12.50
App & Firmware         12.04
Warranty & Repair      11.61
Other                   8.69
Product Enquiry         0.34
Account & Login         0.32
dtype: float64


In [80]:
category_analysis = pd.DataFrame({
    "total_tickets": category_total,
    "repeat_candidates": category_repeat
})

category_analysis["repeat_rate"] = (
    category_analysis["repeat_candidates"]
    / category_analysis["total_tickets"]
)

category_analysis = category_analysis.sort_values(
    "repeat_rate",
    ascending=False
)

category_analysis

,total_tickets,repeat_candidates,repeat_rate
category,,,
Delivery & Shipping,2134,326,0.152765
Audio Quality,792,116,0.146465
Charging & Battery,955,137,0.143455
Returns & Refunds,1197,161,0.134503
Connectivity,1131,150,0.132626
Billing & Payments,1624,203,0.125000
App & Firmware,822,99,0.120438
Warranty & Repair,620,72,0.116129
Other,1691,147,0.086931


In [81]:
category_cost = (
    repeat_unique
    .groupby("category")["repeat_contact_cost"]
    .sum()
)

category_analysis["repeat_cost"] = category_cost

category_analysis.sort_values(
    "repeat_cost",
    ascending=False
)

,total_tickets,repeat_candidates,repeat_rate,repeat_cost
category,,,,
Delivery & Shipping,2134,326,0.152765,84230
Billing & Payments,1624,203,0.125000,54180
Returns & Refunds,1197,161,0.134503,41920
Connectivity,1131,150,0.132626,38430
Other,1691,147,0.086931,38220
Charging & Battery,955,137,0.143455,37330
Audio Quality,792,116,0.146465,29690
App & Firmware,822,99,0.120438,24260
Warranty & Repair,620,72,0.116129,18580


In [82]:
product_repeat = (
    repeat_unique
    .groupby("product_sku")
    .size()
    .sort_values(ascending=False)
)

product_total = (
    final_tickets
    .groupby("product_sku")
    .size()
)

product_analysis = pd.DataFrame({
    "total_tickets": product_total,
    "repeat_candidates": product_repeat
}).fillna(0)

product_analysis["repeat_rate"] = (
    product_analysis["repeat_candidates"]
    / product_analysis["total_tickets"]
)

product_analysis = product_analysis.sort_values(
    "repeat_candidates",
    ascending=False
)

product_analysis

,total_tickets,repeat_candidates,repeat_rate
product_sku,,,
VA-EB-PL2,3401,398,0.117024
VA-EB-PL1,1518,182,0.119895
VA-SW-NX2,1253,174,0.138867
VA-EB-AIR,1030,113,0.109709
VA-SP-MINI,845,109,0.128994
VA-HP-ST3,915,98,0.107104
VA-SW-FIT,660,70,0.106061
VA-SP-ORB,503,59,0.117296
VA-HP-ST2,423,48,0.113475


In [83]:
category_product_repeat = (
    repeat_unique
    .groupby(["category", "product_sku"])
    .size()
    .sort_values(ascending=False)
)

print(category_product_repeat.head(20))

category             product_sku
Delivery & Shipping  VA-EB-PL2      71
Billing & Payments   VA-EB-PL2      58
Charging & Battery   VA-EB-PL2      48
Returns & Refunds    VA-EB-PL2      48
Audio Quality        VA-EB-PL2      47
Connectivity         VA-EB-PL2      45
Other                VA-EB-PL2      44
Delivery & Shipping  VA-SW-NX2      43
Charging & Battery   VA-EB-PL1      35
Delivery & Shipping  VA-EB-PL1      34
                     VA-SP-MINI     31
                     VA-EB-AIR      27
Billing & Payments   VA-SW-NX2      26
App & Firmware       VA-EB-PL2      24
Other                VA-SW-NX2      24
Billing & Payments   VA-EB-PL1      22
Delivery & Shipping  VA-HP-ST3      22
Warranty & Repair    VA-SW-NX2      21
Charging & Battery   VA-EB-AIR      21
Returns & Refunds    VA-SW-NX2      20
dtype: int64


In [84]:
products = pd.read_csv("../data/products.csv")

In [85]:
print("Duplicate product SKUs:",
      products["sku"].duplicated().sum())

Duplicate product SKUs: 0


In [86]:
final_tickets = final_tickets.merge(
    products[["sku", "product_name"]],
    left_on="product_sku",
    right_on="sku",
    how="left"
).drop(columns="sku")

In [87]:
repeat_unique = repeat_unique.merge(
    products[["sku", "product_name"]],
    left_on="product_sku",
    right_on="sku",
    how="left"
).drop(columns="sku")

In [88]:
print(
    final_tickets[
        ["product_sku", "product_name"]
    ].drop_duplicates().sort_values("product_sku")
)

     product_sku                   product_name
2840  VA-AC-CASE    Pulse Charging Case (spare)
24     VA-AC-CBL            USB-C Braided Cable
3     VA-AC-CH65                65W GaN Charger
0      VA-EB-AIR                AirLite Earbuds
4      VA-EB-PL1    Pulse True Wireless Earbuds
232    VA-EB-PL2  Pulse 2 True Wireless Earbuds
1      VA-HP-ST2   Strata 2 Over-Ear Headphones
6      VA-HP-ST3   Strata 3 Over-Ear Headphones
7      VA-NB-ARC                   Arc Neckband
373   VA-SP-MINI             Orbit Mini Speaker
19     VA-SP-ORB            Orbit Smart Speaker
2      VA-SW-FIT                  Nexa Fit Band
11     VA-SW-NX1                Nexa Smartwatch
182    VA-SW-NX2              Nexa 2 Smartwatch


In [89]:
pl2_repeats = repeat_unique[
    repeat_unique["product_sku"] == "VA-EB-PL2"
]

In [90]:
pl2_repeats[
    [
        "ticket_id",
        "category",
        "customer_message"
    ]
].head(30)

,ticket_id,category,customer_message
117,TK-241588,Warranty & Repair,hello\nwarranty claim pending for 9 days\n??
146,TK-241826,Warranty & Repair,hii\nyet again.\nwarranty claim pending for 14...
167,TK-242200,Audio Quality,"Dear team,\nOrdered Pulse2 recently. Mic not ..."
309,TK-243503,Returns & Refunds,hello\nwrong product delivered\nhelp
320,TK-243661,Delivery & Shipping,"dear sir/madam,\n\ni am writing with reference..."
323,TK-243696,Audio Quality,This is very disappointing. Writing again beca...
324,TK-244109,Billing & Payments,product: pulse2\norder: vr898792\npurchased: 2...
327,TK-244019,Returns & Refunds,I am losing patience. Again the same thing. Or...
334,TK-244354,Delivery & Shipping,hello\nfollowing up on my earlier complaint - ...
344,TK-243907,Returns & Refunds,Really frustrating. Got pulse 2 buds from Amaz...


In [91]:
theme_df = final_tickets.copy()

In [92]:
theme_df["message_text"] = (
    theme_df["customer_message"]
    .fillna("")
    .astype(str)
    .str.replace(r"\\n", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

In [93]:
theme_df[
    ["ticket_id", "category", "product_sku", "message_text"]
].head(10)

,ticket_id,category,product_sku,message_text
0,TK-240001,Account & Login,VA-EB-AIR,hii cannot login to my account hello??
1,TK-240002,Audio Quality,VA-HP-ST2,hello buzzing sound from the speaker ??
2,TK-240003,Billing & Payments,VA-SW-FIT,the page failed after i paid and now nothing s...
3,TK-240004,Delivery & Shipping,VA-AC-CH65,honestly regrettign this purchase. the gan cha...
4,TK-240005,Delivery & Shipping,VA-EB-PL1,[ivr transcript] hi the parcel looked like it ...
5,TK-240006,Returns & Refunds,VA-EB-AIR,got airlite earbuds from amazon 3 weeks back. ...
6,TK-240007,Charging & Battery,VA-HP-ST3,"battery drains very fast, barely lasts 2 hours..."
7,TK-240008,Delivery & Shipping,VA-NB-ARC,"hi team, i bought arc neckband on 27-12-2024. ..."
8,TK-240009,Delivery & Shipping,VA-NB-ARC,"hi there, got my arc from vireo.in last week. ..."
9,TK-240010,Billing & Payments,VA-HP-ST3,the page failed after i paid & now nothing sho...


In [94]:
empty_messages = (
    theme_df["message_text"].str.len() == 0
).sum()

print("Empty messages:", empty_messages)

Empty messages: 0


In [95]:
theme_df = theme_df[
    theme_df["message_text"].str.len() > 10
].copy()

## Creating embeddings

In [96]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [97]:
test_embeddings = embedding_model.encode(
    theme_df["message_text"].head(5).tolist(),
    show_progress_bar=False
)

print(test_embeddings.shape)

(5, 384)


In [98]:
print("Empty messages:", empty_messages)
print("Embedding shape:", test_embeddings.shape)

Empty messages: 0
Embedding shape: (5, 384)


In [99]:
import numpy as np

messages = theme_df["message_text"].tolist()

embeddings = embedding_model.encode(
    messages,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/186 [00:00<?, ?it/s]

Embedding shape: (11875, 384)


In [100]:
np.save(
    "../data/customer_message_embeddings.npy",
    embeddings
)

In [101]:
import os

os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count())

In [102]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=15,
    random_state=42,
    n_init=10
)

theme_df["theme_cluster"] = kmeans.fit_predict(
    embeddings
)

c:\Users\pavan\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\pavan\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\pavan\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\pavan\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\pavan\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

In [103]:
cluster_counts = (
    theme_df["theme_cluster"]
    .value_counts()
    .sort_index()
)

print(cluster_counts)

theme_cluster
0      496
1      851
2     1462
3      593
4      434
5      620
6     1024
7     1321
8     1197
9      461
10     732
11     685
12    1044
13     538
14     417
Name: count, dtype: int64


In [104]:
for cluster_id in sorted(theme_df["theme_cluster"].unique()):
    print("\n" + "=" * 80)
    print("CLUSTER:", cluster_id)

    examples = (
        theme_df[
            theme_df["theme_cluster"] == cluster_id
        ]["message_text"]
        .sample(
            min(5, (theme_df["theme_cluster"] == cluster_id).sum()),
            random_state=42
        )
    )

    for msg in examples:
        print("-", msg[:300])


CLUSTER: 0
- namaste, order vr900239 (the nexa 2). the pairing light blinks but nothing eevr shows up on my moto. i forgot the device and tried again. not acceptable at this price. i want a replacement. thanks in advance
- hi, order vr881677 (the pulse buds). pairing is not working. please advise. thanks
- dear team, i bought my pulse 2 on 09 aug. pairing is not working. i forgot the device & tried again. nothing changed. need this sorted this week. awaiting response jaspreet kumar
- bluetooth pairing fails every time. vr883930. please resolve aasp
- order vr904336 (pulse2). it connects for a second & then vanishes from the device list. i restarted the phone. nothing changed. waiting for youur reply. regards divya

CLUSTER: 1
- got pulse earbuds from amazoon around diwali. blueooth keeps cutting out. i reset both devices. nothing changed. please advise. awaitting response
- dear sir/madam, i am writing w/ reference to my order of pulse earbuds (vr899680) paced on 21 jan. return pickup

In [105]:
cluster_summary = (
    theme_df
    .groupby("theme_cluster")
    .agg(
        ticket_count=("ticket_id", "count"),
        avg_csat=("csat_score", "mean")
    )
    .sort_values("ticket_count", ascending=False)
)

cluster_summary

,ticket_count,avg_csat
theme_cluster,,
2,1462,3.36124
7,1321,3.336207
8,1197,3.426716
12,1044,3.458244
6,1024,3.020045
1,851,3.290761
10,732,3.2125
11,685,3.281553
5,620,3.083333


In [106]:
delivery_df = theme_df[
    theme_df["category"] == "Delivery & Shipping"
].copy()

print(delivery_df.shape)


(2134, 31)


In [107]:
delivery_embeddings = embedding_model.encode(
    delivery_df["message_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(delivery_embeddings.shape)

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

(2134, 384)


In [108]:
delivery_kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

delivery_df["theme_cluster"] = (
    delivery_kmeans.fit_predict(delivery_embeddings)
)

c:\Users\pavan\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=9.
  warnings.warn(


In [109]:
for cluster_id in sorted(
    delivery_df["theme_cluster"].unique()
):
    print("\n" + "=" * 70)
    print("CLUSTER:", cluster_id)

    examples = delivery_df[
        delivery_df["theme_cluster"] == cluster_id
    ]["message_text"].sample(
        min(8, (delivery_df["theme_cluster"] == cluster_id).sum()),
        random_state=42
    )

    for msg in examples:
        print("-", msg[:300])


CLUSTER: 0
- hello ji screen has a crack before i even switched it on nexa 2 vr905431 kindly do the needful
- screen has a crack before i even switched it on
- bhai product arrived damaged
- honestly regretting this purchase. this is regarding spare case. i haven't received my order. i already checked with neighbours. fix this or i am posting on twitter.
- hi there's a dent on the case straight out of the box hello??
- hi received damaged product ??
- teh box sys orbbit mini but what's inside is not what i paid for
- dear sir/madam, i am writing with reference to my order of orbit speaker (vr884160) placed on january 17. it has been 15 days and i have nothing in hand. i have checked with neighbours. i request you to kindly arrange a replacement. sincerely, manish ghosh

CLUSTER: 1
- this is very disappointing. this is regarding pulse 2. got a different colour tahn ordered. i already rechecked my order. this is teh last time i buy from you.
- not acceptable at this price. this is regar

In [110]:
delivery_kmeans_8 = KMeans(
    n_clusters=8,
    random_state=42,
    n_init=10
)

delivery_df["theme_cluster_8"] = (
    delivery_kmeans_8.fit_predict(delivery_embeddings)
)

c:\Users\pavan\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=9.
  warnings.warn(


In [111]:
delivery_df["theme_cluster_8"].value_counts().sort_index()

theme_cluster_8
0    259
1    278
2    188
3    225
4    155
5    112
6    302
7    615
Name: count, dtype: int64

In [112]:
for cluster_id in sorted(delivery_df["theme_cluster_8"].unique()):
    print("\n" + "=" * 70)
    print("CLUSTER:", cluster_id)

    examples = delivery_df[
        delivery_df["theme_cluster_8"] == cluster_id
    ]["message_text"].sample(
        min(6, (delivery_df["theme_cluster_8"] == cluster_id).sum()),
        random_state=42
    )

    for msg in examples:
        print("-", msg[:300])


CLUSTER: 0
- hey, i bought nexa 2 on 23/05. package not delivered even after 18 days. please help.
- hi team, ordered nexa 2 in april. nobody came for the pickup. can someone fix this? thnak you
- dear sir/madam, i am writing with reference to my order of the nexa 2 (vr893431) placed on 26 jan. the courier marked it delivered but nobody in my house got anything. i have called the courier. i request u to provide a resolution within 3 working days. regards, shreya mehta
- product: my nexa 2 watch purchased: 06-06-2025 issue: my order has not been delivered yet tried: checked with neighbours expected: fix
- to the vireo customer care team, i am writing with reference to my order of my nexa 2 wacth placed on 04/11. typo in the flat number, courier will never find it. i have tried editing in app. i request you to look into this matter at the earliest. yours faithfully, reyansh khan
- i am losing patience. ordered nexa 2 2 weeks ago. there's a dent on the case straight out of the box. i alr

## 9.3 Finding: clusters still mixed actionable complaint types

In [113]:
delivery_themes = {
    "delivery_delay": 
        "order is delayed, not delivered, or customer is still waiting",

    "tracking_issue":
        "tracking is not updating, shipment status is stuck, or out for delivery for too long",

    "marked_delivered_not_received":
        "courier says delivered but customer did not receive the package",

    "wrong_product":
        "customer received the wrong product, wrong item, or wrong colour/variant",

    "damaged_product":
        "product or package arrived damaged, cracked, dented, or broken",

    "pickup_issue":
        "pickup, return pickup, reshipment, or courier pickup did not happen",

    "address_issue":
        "customer needs to change or correct the delivery address"
}

In [114]:
theme_names = list(delivery_themes.keys())
theme_descriptions = list(delivery_themes.values())

theme_embeddings = embedding_model.encode(
    theme_descriptions,
    normalize_embeddings=True
)

print(theme_embeddings.shape)

(7, 384)


In [115]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    delivery_embeddings,
    theme_embeddings
)

delivery_df["theme_score"] = similarities.max(axis=1)

delivery_df["theme"] = [
    theme_names[i]
    for i in similarities.argmax(axis=1)
]

In [116]:
delivery_df["theme"].value_counts()

theme
marked_delivered_not_received    656
delivery_delay                   440
tracking_issue                   363
damaged_product                  353
wrong_product                    220
pickup_issue                      75
address_issue                     27
Name: count, dtype: int64

In [117]:
for theme in delivery_df["theme"].unique():

    print("\n" + "=" * 70)
    print("THEME:", theme)

    examples = delivery_df[
        delivery_df["theme"] == theme
    ]["message_text"].sample(
        min(6, (delivery_df["theme"] == theme).sum()),
        random_state=42
    )

    for msg in examples:
        print("-", msg[:300])
        


THEME: pickup_issue
- namaste, got the airlite buds from vireo.in a month ago. nobody came for the pickup. i rescheduled pickup twice. how do i get this fixed? thank you
- nobody came for the pickup. vr909504
- to the vireo customer care team, i am writing with reference to my order of my orbit (vr904445) placed on 07/05. return pickup has not happened. i have rescheduled pickup twice. i request you to process a refund to my original payment method. yours faithfully, abhishek saxena
- honestly regrettign this purchase. the gan charger purchased in december, order vr909673. paid on 20-12-2024, still waiting for something to show up. i already checked with neighbours. escalate this to someone senior.
- dear sir/madam, this was supposedly sorted by your team last time. i am writing with reference to my order of the mini speaker placed on 02 oct. pickup scheduled but no one showed up. i have called courier. i request you to provide a resolution within 3 working days. sincerely, isha chauh

In [118]:
print(delivery_df["theme_score"].describe())

count    2134.000000
mean        0.454305
std         0.120159
min         0.113103
25%         0.364440
50%         0.445475
75%         0.532992
max         0.793449
Name: theme_score, dtype: float64


In [119]:
print(
    delivery_df.sort_values("theme_score")[
        [
            "ticket_id",
            "category",
            "theme",
            "theme_score",
            "message_text"
        ]
    ].head(20).to_string(index=False)
)

ticket_id            category                         theme  theme_score                                                                                                                                                                                                                                                              message_text
TK-242856 Delivery & Shipping                 wrong_product     0.113103                                                                                                                                                                      the box says the airlite buds but what's isnide is not what i pad for, vr894511, please resolve asap
TK-252098 Delivery & Shipping               damaged_product     0.165558                                                                                                                                                                                           [ivr transcript] hey my oder has not bene delivred yet the stra

In [120]:
print(
    delivery_df.sort_values(
        "theme_score",
        ascending=False
    )[
        [
            "theme",
            "theme_score",
            "message_text"
        ]
    ].head(20).to_string(index=False)
)

                        theme  theme_score                                                                           message_text
              damaged_product     0.793449                                                                product arrived damaged
              damaged_product     0.793449                                                                product arrived damaged
              damaged_product     0.793449                                                                product arrived damaged
              damaged_product     0.793449                                                                product arrived damaged
              damaged_product     0.793449                                                                product arrived damaged
                 pickup_issue     0.790293                                                         return pickup has not happened
                 pickup_issue     0.790293                                                

In [121]:
low_score = (
    delivery_df
    .sort_values("theme_score")
    [
        [
            "ticket_id",
            "theme",
            "theme_score",
            "message_text"
        ]
    ]
    .head(20)
)

print(low_score.to_string(index=False))

ticket_id                         theme  theme_score                                                                                                                                                                                                                                                              message_text
TK-242856                 wrong_product     0.113103                                                                                                                                                                      the box says the airlite buds but what's isnide is not what i pad for, vr894511, please resolve asap
TK-252098               damaged_product     0.165558                                                                                                                                                                                           [ivr transcript] hey my oder has not bene delivred yet the strata 3 vr903680 ??
TK-243770                 address_issue    

In [122]:
high_score = (
    delivery_df
    .sort_values("theme_score", ascending=False)
    [
        [
            "ticket_id",
            "theme",
            "theme_score",
            "message_text"
        ]
    ]
    .head(20)
)

print(high_score.to_string(index=False))

ticket_id                         theme  theme_score                                                                           message_text
TK-243528               damaged_product     0.793449                                                                product arrived damaged
TK-242687               damaged_product     0.793449                                                                product arrived damaged
TK-253357               damaged_product     0.793449                                                                product arrived damaged
TK-242398               damaged_product     0.793449                                                                product arrived damaged
TK-247715               damaged_product     0.793449                                                                product arrived damaged
TK-244099                  pickup_issue     0.790293                                                         return pickup has not happened
TK-252979           

## 11. Weekly Support Trends

In [123]:
final_tickets["week"] = (
    final_tickets["created_at"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

In [124]:
final_tickets[
    ["created_at", "week"]
].head()

,created_at,week
0,2025-01-01 09:48:00,2024-12-30
1,2025-01-01 13:24:00,2024-12-30
2,2025-01-01 13:55:00,2024-12-30
3,2025-01-01 15:13:00,2024-12-30
4,2025-01-01 20:24:00,2024-12-30


In [125]:
weekly_volume = (
    final_tickets
    .groupby("week")
    .size()
    .reset_index(name="ticket_count")
)

weekly_volume.head(10)

,week,ticket_count
0,2024-12-30,47
1,2025-01-06,54
2,2025-01-13,59
3,2025-01-20,82
4,2025-01-27,67
5,2025-02-03,78
6,2025-02-10,68
7,2025-02-17,83
8,2025-02-24,80
9,2025-03-03,96


In [126]:
weekly_volume = (
    final_tickets
    .groupby("week")
    .size()
    .reset_index(name="ticket_count")
)

weekly_volume.head(10)

,week,ticket_count
0,2024-12-30,47
1,2025-01-06,54
2,2025-01-13,59
3,2025-01-20,82
4,2025-01-27,67
5,2025-02-03,78
6,2025-02-10,68
7,2025-02-17,83
8,2025-02-24,80
9,2025-03-03,96


In [127]:
print(
    weekly_volume.sort_values(
        "ticket_count",
        ascending=False
    ).head(10)
)

         week  ticket_count
46 2025-11-17           240
47 2025-11-24           232
71 2026-05-11           231
48 2025-12-01           230
44 2025-11-03           228
43 2025-10-27           224
41 2025-10-13           216
42 2025-10-20           215
45 2025-11-10           213
50 2025-12-15           206


In [128]:
weekly_category = (
    final_tickets
    .groupby(["week", "category"])
    .size()
    .reset_index(name="ticket_count")
)

In [129]:
weekly_category[
    weekly_category["week"] >= weekly_category["week"].max() - pd.Timedelta(weeks=8)
].sort_values(
    ["week", "ticket_count"],
    ascending=[False, False]
).head(20)

,week,category,ticket_count
862,2026-06-29,Other,8
858,2026-06-29,Billing & Payments,7
861,2026-06-29,Delivery & Shipping,6
864,2026-06-29,Returns & Refunds,6
856,2026-06-29,App & Firmware,4
857,2026-06-29,Audio Quality,3
859,2026-06-29,Charging & Battery,3
860,2026-06-29,Connectivity,3
863,2026-06-29,Product Enquiry,3
865,2026-06-29,Warranty & Repair,2


In [130]:
final_tickets["quarter_period"] = pd.cut(
    final_tickets["created_at"],
    bins=[
        pd.Timestamp("2026-01-01"),
        pd.Timestamp("2026-04-01"),
        pd.Timestamp("2026-07-01")
    ],
    labels=["Jan-Mar 2026", "Apr-Jun 2026"],
    right=False
)

In [131]:
period_category = pd.crosstab(
    final_tickets["category"],
    final_tickets["quarter_period"]
)

period_category


quarter_period,Jan-Mar 2026,Apr-Jun 2026
category,,
Account & Login,71,69
App & Firmware,187,197
Audio Quality,193,178
Billing & Payments,315,286
Charging & Battery,234,214
Connectivity,261,240
Delivery & Shipping,384,470
Other,324,346
Product Enquiry,117,110


In [132]:
period_category["pct_change"] = (
    (
        period_category["Apr-Jun 2026"]
        - period_category["Jan-Mar 2026"]
    )
    / period_category["Jan-Mar 2026"]
    * 100
)

period_category.sort_values(
    "pct_change",
    ascending=False
)

quarter_period,Jan-Mar 2026,Apr-Jun 2026,pct_change
category,,,
Delivery & Shipping,384,470,22.395833
Warranty & Repair,125,134,7.200000
Other,324,346,6.790123
App & Firmware,187,197,5.347594
Account & Login,71,69,-2.816901
Product Enquiry,117,110,-5.982906
Returns & Refunds,241,223,-7.468880
Audio Quality,193,178,-7.772021
Connectivity,261,240,-8.045977


In [133]:
final_tickets["repeat_candidate"] = (
    final_tickets["ticket_id"].isin(
        repeat_unique["ticket_id"]
    )
)

In [134]:
weekly_repeat = (
    final_tickets
    .groupby("week")
    .agg(
        total_tickets=("ticket_id", "count"),
        repeat_candidates=("repeat_candidate", "sum")
    )
)

weekly_repeat["repeat_rate"] = (
    weekly_repeat["repeat_candidates"]
    / weekly_repeat["total_tickets"]
)

weekly_repeat.tail(12)


,total_tickets,repeat_candidates,repeat_rate
week,,,
2026-04-13,201,26,0.129353
2026-04-20,189,26,0.137566
2026-04-27,174,24,0.137931
2026-05-04,157,25,0.159236
2026-05-11,231,23,0.099567
2026-05-18,204,24,0.117647
2026-05-25,186,19,0.102151
2026-06-01,171,26,0.152047
2026-06-08,203,22,0.108374


In [135]:
delivery_weekly_repeat = (
    final_tickets[
        final_tickets["category"] == "Delivery & Shipping"
    ]
    .groupby("week")
    .agg(
        total_tickets=("ticket_id", "count"),
        repeat_candidates=("repeat_candidate", "sum")
    )
)

delivery_weekly_repeat["repeat_rate"] = (
    delivery_weekly_repeat["repeat_candidates"]
    / delivery_weekly_repeat["total_tickets"]
)

delivery_weekly_repeat.tail(12)

,total_tickets,repeat_candidates,repeat_rate
week,,,
2026-04-13,41,9,0.219512
2026-04-20,34,4,0.117647
2026-04-27,33,7,0.212121
2026-05-04,25,8,0.320000
2026-05-11,52,8,0.153846
2026-05-18,38,5,0.131579
2026-05-25,40,8,0.200000
2026-06-01,30,10,0.333333
2026-06-08,39,7,0.179487


In [136]:
csat_count = final_tickets["csat_score"].notna().sum()

avg_csat = final_tickets["csat_score"].mean()

print("CSAT responses:", csat_count)
print("Average CSAT:", round(avg_csat, 2))

CSAT responses: 5269
Average CSAT: 3.32


In [137]:
category_csat = (
    final_tickets
    .groupby("category")["csat_score"]
    .agg(["count", "mean"])
    .sort_values("mean")
)

category_csat

,count,mean
category,,
Delivery & Shipping,961,3.034339
Returns & Refunds,521,3.053743
Charging & Battery,441,3.0839
Warranty & Repair,278,3.093525
Audio Quality,326,3.159509
Connectivity,484,3.42562
Billing & Payments,727,3.452545
App & Firmware,377,3.453581
Other,743,3.56393


In [138]:
repeat_csat = (
    final_tickets
    .groupby("repeat_candidate")["csat_score"]
    .agg(["count", "mean"])
)

repeat_csat

,count,mean
repeat_candidate,,
False,4661,3.431667
True,608,2.422697


In [139]:
category_csat_repeat = (
    final_tickets
    .groupby(["category", "repeat_candidate"])["csat_score"]
    .agg(["count", "mean"])
)

category_csat_repeat

count      mean
category            repeat_candidate                 
Account & Login     False               148      3.75
                    True                  1       3.0
App & Firmware      False               333  3.606607
                    True                 44  2.295455
Audio Quality       False               279  3.286738
                    True                 47  2.404255
Billing & Payments  False               629  3.586645
                    True                 98  2.591837
Charging & Battery  False               378  3.206349
                    True                 63  2.349206
Connectivity        False               422  3.559242
                    True                 62  2.516129
Delivery & Shipping False               818  3.161369
                    True                143  2.307692
Other               False               689  3.619739
                    True                 54  2.851852
Product Enquiry     False               262  3.950382
                    True                  0       NaN
Returns & Refunds   False               460   3.16087
                    True                 61  2.245902
Warranty & Repair   False               243  3.222222
                    True                 35       2.2

In [140]:
csat_pivot = (
    category_csat_repeat
    .reset_index()
    .pivot(
        index="category",
        columns="repeat_candidate",
        values="mean"
    )
)

csat_pivot["csat_gap"] = (
    csat_pivot[False] - csat_pivot[True]
)

csat_pivot.sort_values(
    "csat_gap",
    ascending=False
)


repeat_candidate,False,True,csat_gap
category,,,
App & Firmware,3.606607,2.295455,1.311152
Connectivity,3.559242,2.516129,1.043113
Warranty & Repair,3.222222,2.2,1.022222
Billing & Payments,3.586645,2.591837,0.994809
Returns & Refunds,3.16087,2.245902,0.914968
Audio Quality,3.286738,2.404255,0.882483
Charging & Battery,3.206349,2.349206,0.857143
Delivery & Shipping,3.161369,2.307692,0.853677
Other,3.619739,2.851852,0.767887


## CSAT difference by category

In [141]:
category_csat_repeat = (
    final_tickets
    .groupby(["category", "repeat_candidate"])["csat_score"]
    .agg(["count", "mean"])
)

csat_pivot = (
    category_csat_repeat
    .reset_index()
    .pivot(
        index="category",
        columns="repeat_candidate",
        values="mean"
    )
)

csat_pivot["csat_gap"] = (
    csat_pivot[False] - csat_pivot[True]
)

csat_pivot.sort_values(
    "csat_gap",
    ascending=False
)

repeat_candidate,False,True,csat_gap
category,,,
App & Firmware,3.606607,2.295455,1.311152
Connectivity,3.559242,2.516129,1.043113
Warranty & Repair,3.222222,2.2,1.022222
Billing & Payments,3.586645,2.591837,0.994809
Returns & Refunds,3.16087,2.245902,0.914968
Audio Quality,3.286738,2.404255,0.882483
Charging & Battery,3.206349,2.349206,0.857143
Delivery & Shipping,3.161369,2.307692,0.853677
Other,3.619739,2.851852,0.767887


## the refund/replacement impact

In [142]:
refund_summary = {
    "refund_tickets": final_tickets["refund_amount_inr"].notna().sum(),
    "refund_total": final_tickets["refund_amount_inr"].sum(),
    "replacement_tickets": (
        final_tickets["replacement_issued"] == "Y"
    ).sum(),
    "refund_replacement_conflicts": (
        final_tickets["refund_replacement_conflict"]
    ).sum()
}

refund_summary

{'refund_tickets': 2105,
 'refund_total': 5992919.0,
 'replacement_tickets': 1202,
 'refund_replacement_conflicts': 4}

In [143]:
category_refund = (
    final_tickets
    .groupby("category")
    .agg(
        tickets=("ticket_id", "count"),
        refund_tickets=("refund_amount_inr", "count"),
        refund_amount=("refund_amount_inr", "sum"),
        replacements=("replacement_issued", lambda x: (x == "Y").sum())
    )
)

category_refund["refund_rate"] = (
    category_refund["refund_tickets"]
    / category_refund["tickets"]
)

category_refund.sort_values(
    "refund_amount",
    ascending=False
)

,tickets,refund_tickets,refund_amount,replacements,refund_rate
category,,,,,
Returns & Refunds,1197,645,1926282.0,74,0.538847
Billing & Payments,1624,670,1793578.0,0,0.412562
Other,1691,373,1099435.0,83,0.220580
Delivery & Shipping,2134,252,721253.0,360,0.118088
Warranty & Repair,620,52,166187.0,178,0.083871
Audio Quality,792,32,87454.0,166,0.040404
Charging & Battery,955,34,82620.0,207,0.035602
Connectivity,1131,28,75691.0,89,0.024757
Product Enquiry,593,9,21319.0,0,0.015177


In [144]:
current_refund = final_tickets[
    final_tickets["source_system"] == "helpdesk"
]

print("Current-helpdesk refund tickets:",
      current_refund["refund_amount_inr"].notna().sum())

print("Current-helpdesk refund amount: ₹",
      current_refund["refund_amount_inr"].sum())

print("Current-helpdesk replacements:",
      (current_refund["replacement_issued"] == "Y").sum())

print("Current-helpdesk refund+replacement conflicts:",
      current_refund["refund_replacement_conflict"].sum())

Current-helpdesk refund tickets: 1524
Current-helpdesk refund amount: ₹ 4225922.0
Current-helpdesk replacements: 886
Current-helpdesk refund+replacement conflicts: 2


In [145]:
policy_tickets = final_tickets[
    final_tickets["created_at"] >= pd.Timestamp("2025-04-01")
].copy()

print("Policy-period tickets:", len(policy_tickets))

print(
    "Policy-period SLA breaches:",
    policy_tickets["sla_breach"].sum()
)

print(
    "Policy-period SLA breach rate:",
    policy_tickets["sla_breach"].mean() * 100
)

print(
    "Policy-period SLA credit exposure: ₹",
    policy_tickets["sla_breach"].sum() * 350
)

Policy-period tickets: 10867
Policy-period SLA breaches: 957
Policy-period SLA breach rate: 8.806478328885618
Policy-period SLA credit exposure: ₹ 334950


## 12. KPI Summary

Core business metrics calculated from the cleaned ticket dataset.
Policy-based SLA metrics use tickets created from 2025-04-01 onward.
Repeat-contact figures are candidate metrics based on a structured proxy and are not treated as confirmed ground truth.
Legacy monetary values are not combined with current-helpdesk rupee values because the supplied policy does not specify the legacy monetary unit.

In [146]:
kpi_summary = pd.DataFrame({
    "Metric": [
        "Clean tickets",
        "Policy-period SLA breaches",
        "Policy-period SLA breach rate",
        "SLA credit exposure",
        "Current-helpdesk transfers",
        "Current-helpdesk transfer cost",
        "Repeat-contact candidates",
        "Repeat-contact candidate rate",
        "Repeat-contact candidate cost",
        "CSAT responses",
        "Average CSAT",
        "Current-helpdesk refunds",
        "Current-helpdesk refund amount",
        "Current-helpdesk replacements",
        "Refund + replacement conflicts"
    ],
    "Value": [
        len(final_tickets),
        policy_tickets["sla_breach"].sum(),
        policy_tickets["sla_breach"].mean() * 100,
        policy_tickets["sla_breach"].sum() * 350,
        current_transfers,
        current_transfer_cost,
        len(repeat_unique),
        final_tickets["repeat_candidate"].mean() * 100,
        repeat_unique["repeat_contact_cost"].sum(),
        final_tickets["csat_score"].notna().sum(),
        final_tickets["csat_score"].mean(),
        current_refund["refund_amount_inr"].notna().sum(),
        current_refund["refund_amount_inr"].sum(),
        (current_refund["replacement_issued"] == "Y").sum(),
        current_refund["refund_replacement_conflict"].sum()
    ]
})

kpi_summary

,Metric,Value
0,Clean tickets,1.187500e+04
1,Policy-period SLA breaches,9.570000e+02
2,Policy-period SLA breach rate,8.806478e+00
3,SLA credit exposure,3.349500e+05
4,Current-helpdesk transfers,8.680000e+02
5,Current-helpdesk transfer cost,2.647400e+05
6,Repeat-contact candidates,1.414000e+03
7,Repeat-contact candidate rate,1.190737e+01
8,Repeat-contact candidate cost,3.675000e+05
9,CSAT responses,5.269000e+03


## 13. Weekly Support Digest

In [147]:
weekly_kpis = (
    final_tickets
    .groupby("week")
    .agg(
        tickets=("ticket_id", "count"),
        repeat_candidates=("repeat_candidate", "sum"),
        sla_breaches=("sla_breach", "sum"),
        avg_csat=("csat_score", "mean"),
        transfers=("transfers", "sum"),
        refunds=("refund_amount_inr", "count"),
        refund_amount=("refund_amount_inr", "sum")
    )
)

In [148]:
weekly_kpis["repeat_rate"] = (
    weekly_kpis["repeat_candidates"]
    / weekly_kpis["tickets"]
)

weekly_kpis["sla_breach_rate"] = (
    weekly_kpis["sla_breaches"]
    / weekly_kpis["tickets"]
)

weekly_kpis["sla_credit_exposure"] = (
    weekly_kpis["sla_breaches"] * 350
)

In [149]:
weekly_kpis.tail(12)

,tickets,repeat_candidates,sla_breaches,avg_csat,transfers,refunds,refund_amount,repeat_rate,sla_breach_rate,sla_credit_exposure
week,,,,,,,,,,
2026-04-13,201,26,12,3.382716,19,36,98142.0,0.129353,0.059701,4200
2026-04-20,189,26,15,3.269231,8,23,48428.0,0.137566,0.079365,5250
2026-04-27,174,24,19,3.294872,22,23,71084.0,0.137931,0.109195,6650
2026-05-04,157,25,13,3.112903,13,27,84562.0,0.159236,0.082803,4550
2026-05-11,231,23,19,3.327103,23,37,97481.0,0.099567,0.082251,6650
2026-05-18,204,24,20,3.505747,15,32,87979.0,0.117647,0.098039,7000
2026-05-25,186,19,24,3.277108,26,22,73937.0,0.102151,0.129032,8400
2026-06-01,171,26,15,3.186441,12,26,73729.0,0.152047,0.087719,5250
2026-06-08,203,22,21,3.202247,20,30,77355.0,0.108374,0.103448,7350


In [150]:
weekly_kpis["ticket_change_pct"] = (
    weekly_kpis["tickets"]
    .pct_change() * 100
)

In [151]:
weekly_kpis["repeat_change_pct"] = (
    weekly_kpis["repeat_rate"]
    .pct_change() * 100
)

In [152]:
weekly_kpis.tail(12)

,tickets,repeat_candidates,sla_breaches,avg_csat,transfers,refunds,refund_amount,repeat_rate,sla_breach_rate,sla_credit_exposure,ticket_change_pct,repeat_change_pct
week,,,,,,,,,,,,
2026-04-13,201,26,12,3.382716,19,36,98142.0,0.129353,0.059701,4200,-0.985222,25.041459
2026-04-20,189,26,15,3.269231,8,23,48428.0,0.137566,0.079365,5250,-5.970149,6.349206
2026-04-27,174,24,19,3.294872,22,23,71084.0,0.137931,0.109195,6650,-7.936508,0.265252
2026-05-04,157,25,13,3.112903,13,27,84562.0,0.159236,0.082803,4550,-9.770115,15.445860
2026-05-11,231,23,19,3.327103,23,37,97481.0,0.099567,0.082251,6650,47.133758,-37.471861
2026-05-18,204,24,20,3.505747,15,32,87979.0,0.117647,0.098039,7000,-11.688312,18.158568
2026-05-25,186,19,24,3.277108,26,22,73937.0,0.102151,0.129032,8400,-8.823529,-13.172043
2026-06-01,171,26,15,3.186441,12,26,73729.0,0.152047,0.087719,5250,-8.064516,48.845799
2026-06-08,203,22,21,3.202247,20,30,77355.0,0.108374,0.103448,7350,18.713450,-28.723001


In [153]:
weekly_category_counts = (
    final_tickets
    .groupby(["week", "category"])
    .size()
    .reset_index(name="tickets")
)

In [154]:
def top_categories_for_week(week_start, n=5):
    result = weekly_category_counts[
        weekly_category_counts["week"] == week_start
    ].sort_values(
        "tickets",
        ascending=False
    )

    return result.head(n)

In [155]:
latest_week = weekly_kpis.index.max()

print("Latest week:", latest_week)

top_categories_for_week(latest_week)

Latest week: 2026-06-29 00:00:00


,week,category,tickets
862,2026-06-29,Other,8
858,2026-06-29,Billing & Payments,7
861,2026-06-29,Delivery & Shipping,6
864,2026-06-29,Returns & Refunds,6
856,2026-06-29,App & Firmware,4


In [156]:
complete_weeks = weekly_kpis[
    weekly_kpis.index < pd.Timestamp("2026-06-29")
]

latest_complete_week = complete_weeks.index.max()

print("Latest complete week:", latest_complete_week)

Latest complete week: 2026-06-22 00:00:00


In [157]:
top_categories_for_week(latest_complete_week)

,week,category,tickets
850,2026-06-22,Delivery & Shipping,34
848,2026-06-22,Charging & Battery,28
851,2026-06-22,Other,28
853,2026-06-22,Returns & Refunds,23
847,2026-06-22,Billing & Payments,20


In [158]:
current_week = latest_complete_week
previous_week = (
    current_week - pd.Timedelta(weeks=1)
)

current_categories = (
    weekly_category_counts[
        weekly_category_counts["week"] == current_week
    ]
    .set_index("category")["tickets"]
)

previous_categories = (
    weekly_category_counts[
        weekly_category_counts["week"] == previous_week
    ]
    .set_index("category")["tickets"]
)

category_change = pd.DataFrame({
    "current_week": current_categories,
    "previous_week": previous_categories
}).fillna(0)

In [159]:
category_change["change"] = (
    category_change["current_week"]
    - category_change["previous_week"]
)

category_change["change_pct"] = (
    category_change["change"]
    / category_change["previous_week"].replace(0, pd.NA)
) * 100

In [160]:
category_change.sort_values(
    "change",
    ascending=False
)

,current_week,previous_week,change,change_pct
category,,,,
Charging & Battery,28,14,14,100.000000
Returns & Refunds,23,12,11,91.666667
Audio Quality,14,8,6,75.000000
Delivery & Shipping,34,31,3,9.677419
Other,28,25,3,12.000000
Product Enquiry,10,9,1,11.111111
App & Firmware,11,11,0,0.000000
Warranty & Repair,5,5,0,0.000000
Billing & Payments,20,21,-1,-4.761905


In [161]:
final_tickets["current_helpdesk_transfer"] = (
    final_tickets["transfers"]
    .where(final_tickets["source_system"] == "helpdesk", 0)
)

final_tickets["current_helpdesk_refund_amount"] = (
    final_tickets["refund_amount_inr"]
    .where(final_tickets["source_system"] == "helpdesk")
)

In [162]:
weekly_kpis = (
    final_tickets
    .groupby("week")
    .agg(
        tickets=("ticket_id", "count"),
        repeat_candidates=("repeat_candidate", "sum"),
        sla_breaches=("sla_breach", "sum"),
        avg_csat=("csat_score", "mean"),
        transfers=("current_helpdesk_transfer", "sum"),
        refunds=("current_helpdesk_refund_amount", "count"),
        refund_amount=("current_helpdesk_refund_amount", "sum")
    )
)

In [163]:
weekly_kpis["repeat_rate"] = (
    weekly_kpis["repeat_candidates"]
    / weekly_kpis["tickets"]
)

weekly_kpis["sla_breach_rate"] = (
    weekly_kpis["sla_breaches"]
    / weekly_kpis["tickets"]
)

weekly_kpis["sla_credit_exposure"] = (
    weekly_kpis["sla_breaches"] * 350
)

weekly_kpis["ticket_change_pct"] = (
    weekly_kpis["tickets"].pct_change() * 100
)

weekly_kpis["repeat_change_pct"] = (
    weekly_kpis["repeat_rate"].pct_change() * 100
)

In [164]:
current_week = latest_complete_week
previous_week = current_week - pd.Timedelta(weeks=1)

current_kpi = weekly_kpis.loc[current_week]
previous_kpi = weekly_kpis.loc[previous_week]

digest_input = {
    "week": str(current_week.date()),
    "tickets": int(current_kpi["tickets"]),
    "ticket_change_pct": float(current_kpi["ticket_change_pct"]),
    "repeat_candidates": int(current_kpi["repeat_candidates"]),
    "repeat_rate": float(current_kpi["repeat_rate"]),
    "sla_breaches": int(current_kpi["sla_breaches"]),
    "sla_breach_rate": float(current_kpi["sla_breach_rate"]),
    "sla_credit_exposure": float(current_kpi["sla_credit_exposure"]),
    "avg_csat": float(current_kpi["avg_csat"]),
    "transfers": int(current_kpi["transfers"]),
    "refunds": int(current_kpi["refunds"]),
    "refund_amount": float(current_kpi["refund_amount"])
}

digest_input

{'week': '2026-06-22',
 'tickets': 199,
 'ticket_change_pct': 19.16167664670658,
 'repeat_candidates': 23,
 'repeat_rate': 0.11557788944723618,
 'sla_breaches': 15,
 'sla_breach_rate': 0.07537688442211055,
 'sla_credit_exposure': 5250.0,
 'avg_csat': 3.3125,
 'transfers': 21,
 'refunds': 30,
 'refund_amount': 125797.0}

## repeat-contact cost for the week

In [165]:
weekly_repeat_cost = (
    repeat_unique
    .assign(
        week=lambda df: (
            df["created_at"]
            .dt.to_period("W-SUN")
            .dt.start_time
        )
    )
    .groupby("week")["repeat_contact_cost"]
    .sum()
)

weekly_kpis["repeat_contact_cost"] = (
    weekly_repeat_cost.reindex(weekly_kpis.index)
    .fillna(0)
)


In [166]:
weekly_kpis.loc[
    [previous_week, current_week],
    [
        "tickets",
        "repeat_candidates",
        "repeat_rate",
        "repeat_contact_cost",
        "sla_breaches",
        "sla_breach_rate",
        "avg_csat",
        "transfers"
    ]
]

,tickets,repeat_candidates,repeat_rate,repeat_contact_cost,sla_breaches,sla_breach_rate,avg_csat,transfers
week,,,,,,,,
2026-06-15,167,19,0.113772,4690.0,19,0.113772,3.152778,21
2026-06-22,199,23,0.115578,6150.0,15,0.075377,3.3125,21


### top repeat-contact categories for this week

In [167]:
current_repeat_categories = (
    final_tickets[
        (final_tickets["week"] == current_week) &
        (final_tickets["repeat_candidate"])
    ]
    .groupby("category")
    .size()
    .sort_values(ascending=False)
)

current_repeat_categories

category
Other                  7
Delivery & Shipping    4
Returns & Refunds      3
App & Firmware         2
Audio Quality          2
Billing & Payments     2
Charging & Battery     2
Connectivity           1
dtype: int64

### 14. Weekly Digest Generator

In [168]:
def build_weekly_digest(week_start):

    current = weekly_kpis.loc[week_start]

    previous_week = week_start - pd.Timedelta(weeks=1)

    previous = weekly_kpis.loc[previous_week]

    category_counts = (
        weekly_category_counts[
            weekly_category_counts["week"] == week_start
        ]
        .sort_values("tickets", ascending=False)
        .head(5)
    )

    repeat_categories = (
        final_tickets[
            (final_tickets["week"] == week_start) &
            (final_tickets["repeat_candidate"])
        ]
        .groupby("category")
        .size()
        .sort_values(ascending=False)
        .head(5)
    )

    digest = {
        "week": week_start.strftime("%d %b %Y"),
        "tickets": int(current["tickets"]),
        "previous_tickets": int(previous["tickets"]),
        "ticket_change_pct": round(
            (current["tickets"] - previous["tickets"])
            / previous["tickets"] * 100,
            2
        ),
        "repeat_candidates": int(current["repeat_candidates"]),
        "repeat_rate_pct": round(
            current["repeat_rate"] * 100,
            2
        ),
        "repeat_contact_cost": int(
            current["repeat_contact_cost"]
        ),
        "sla_breaches": int(current["sla_breaches"]),
        "sla_breach_rate_pct": round(
            current["sla_breach_rate"] * 100,
            2
        ),
        "sla_credit_exposure": int(
            current["sla_credit_exposure"]
        ),
        "avg_csat": round(
            current["avg_csat"], 2
        ),
        "transfers": int(current["transfers"]),
        "top_categories": category_counts[
            ["category", "tickets"]
        ].to_dict("records"),
        "top_repeat_categories": [
            {
                "category": category,
                "repeat_candidates": int(count)
            }
            for category, count
            in repeat_categories.items()
        ]
    }

    return digest

In [169]:
digest = build_weekly_digest(latest_complete_week)

digest

{'week': '22 Jun 2026',
 'tickets': 199,
 'previous_tickets': 167,
 'ticket_change_pct': 19.16,
 'repeat_candidates': 23,
 'repeat_rate_pct': 11.56,
 'repeat_contact_cost': 6150,
 'sla_breaches': 15,
 'sla_breach_rate_pct': 7.54,
 'sla_credit_exposure': 5250,
 'avg_csat': 3.31,
 'transfers': 21,
 'top_categories': [{'category': 'Delivery & Shipping', 'tickets': 34},
  {'category': 'Charging & Battery', 'tickets': 28},
  {'category': 'Other', 'tickets': 28},
  {'category': 'Returns & Refunds', 'tickets': 23},
  {'category': 'Billing & Payments', 'tickets': 20}],
 'top_repeat_categories': [{'category': 'Other', 'repeat_candidates': 7},
  {'category': 'Delivery & Shipping', 'repeat_candidates': 4},
  {'category': 'Returns & Refunds', 'repeat_candidates': 3},
  {'category': 'App & Firmware', 'repeat_candidates': 2},
  {'category': 'Audio Quality', 'repeat_candidates': 2}]}

In [170]:
def format_digest(digest):

    text = f"""
WEEKLY SUPPORT DIGEST
Week: {digest["week"]}

WHAT CHANGED
Support handled {digest["tickets"]} tickets,
{digest["ticket_change_pct"]:+.1f}% versus the previous week.

REPEAT CONTACT
{digest["repeat_candidates"]} tickets were repeat-contact candidates
({digest["repeat_rate_pct"]:.1f}%).
Estimated candidate repeat-contact cost: ₹{digest["repeat_contact_cost"]:,}.

SLA
{digest["sla_breaches"]} tickets breached first-response SLA
({digest["sla_breach_rate_pct"]:.1f}%).
Estimated SLA credit exposure: ₹{digest["sla_credit_exposure"]:,}.

CUSTOMER EXPERIENCE
Average CSAT: {digest["avg_csat"]:.2f}/5.

OPERATIONS
Transfers: {digest["transfers"]}

TOP CATEGORIES
"""

    for item in digest["top_categories"]:
        text += f'- {item["category"]}: {item["tickets"]} tickets\n'

    text += "\nTOP REPEAT-CONTACT CATEGORIES\n"

    for item in digest["top_repeat_categories"]:
        text += (
            f'- {item["category"]}: '
            f'{item["repeat_candidates"]} repeat candidates\n'
        )

    return text

In [171]:
print(format_digest(digest))


WEEKLY SUPPORT DIGEST
Week: 22 Jun 2026

WHAT CHANGED
Support handled 199 tickets,
+19.2% versus the previous week.

REPEAT CONTACT
23 tickets were repeat-contact candidates
(11.6%).
Estimated candidate repeat-contact cost: ₹6,150.

SLA
15 tickets breached first-response SLA
(7.5%).
Estimated SLA credit exposure: ₹5,250.

CUSTOMER EXPERIENCE
Average CSAT: 3.31/5.

OPERATIONS
Transfers: 21

TOP CATEGORIES
- Delivery & Shipping: 34 tickets
- Charging & Battery: 28 tickets
- Other: 28 tickets
- Returns & Refunds: 23 tickets
- Billing & Payments: 20 tickets

TOP REPEAT-CONTACT CATEGORIES
- Other: 7 repeat candidates
- Delivery & Shipping: 4 repeat candidates
- Returns & Refunds: 3 repeat candidates
- App & Firmware: 2 repeat candidates
- Audio Quality: 2 repeat candidates



In [172]:
current_week_tickets = final_tickets[
    final_tickets["week"] == current_week
].copy()

print(current_week_tickets.shape)

(199, 34)


In [173]:
top_week_categories = (
    current_week_tickets["category"]
    .value_counts()
    .head(5)
)

print(top_week_categories)

category
Delivery & Shipping    34
Other                  28
Charging & Battery     28
Returns & Refunds      23
Billing & Payments     20
Name: count, dtype: int64


In [174]:
complaint_evidence = current_week_tickets[
    current_week_tickets["category"].isin(
        top_week_categories.index
    )
][
    ["ticket_id", "category", "product_name", "customer_message"]
].copy()

complaint_evidence = (
    complaint_evidence
    .groupby("category", group_keys=False)
    .apply(
        lambda x: x.sample(
            min(len(x), 10),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print("Evidence rows:", len(complaint_evidence))

Evidence rows: 50


C:\Users\pavan\AppData\Local\Temp\ipykernel_20268\3586236826.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [175]:
complaint_evidence.head(20)

,ticket_id,category,product_name,customer_message
0,TK-254577,Billing & Payments,Pulse 2 True Wireless Earbuds,"need GST invoice for my order, i want my money..."
1,TK-254800,Billing & Payments,Nexa 2 Smartwatch,hi sir\ni was charged twice for one order\nurgent
2,TK-254774,Billing & Payments,Pulse True Wireless Earbuds,[IVR transcript] Honestly regretting this purc...
3,TK-254594,Billing & Payments,Orbit Smart Speaker,Got Orbit speaker from vireo.in 2 weeks ago. M...
4,TK-254683,Billing & Payments,Pulse 2 True Wireless Earbuds,my bank says Rs 4999 went to you but your site...
5,TK-254646,Billing & Payments,Strata 3 Over-Ear Headphones,[IVR transcript] pathetic experience honestly....
6,TK-254736,Billing & Payments,Pulse 2 True Wireless Earbuds,"Dear Sir/Madam,\n\nRaised this last month and ..."
7,TK-254614,Billing & Payments,Nexa Smartwatch,HELO\nTHE INVOICE PDF LINK GIVES A 404
8,TK-254815,Billing & Payments,Nexa Fit Band,I am losing patience. This is regarding Nexa F...
9,TK-254778,Billing & Payments,Nexa 2 Smartwatch,"paid via UPI, amount deducted, no order confir..."


In [176]:
import requests

response = requests.get("http://localhost:11434/api/tags")

print(response.status_code)
print(response.json())

200
{'models': [{'name': 'qwen3:latest', 'model': 'qwen3:latest', 'modified_at': '2026-08-03T13:08:38.0485936+05:30', 'size': 5225388164, 'digest': '500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen3', 'families': ['qwen3'], 'parameter_size': '8.2B', 'quantization_level': 'Q4_K_M', 'context_length': 40960, 'embedding_length': 4096}, 'capabilities': ['completion', 'tools', 'thinking']}, {'name': 'llama3.2:3b', 'model': 'llama3.2:3b', 'modified_at': '2026-07-27T23:51:44.0501483+05:30', 'size': 2019393189, 'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '3.2B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 3072}, 'capabilities': ['completion', 'tools']}, {'name': 'gemma3:4b', 'model': 'gemma3:4b', 'modified_at': '2026-06-25T23:14:59.69879

In [177]:
MODEL_NAME = "gemma3:4b"

In [178]:
evidence_text = "\n\n".join(
    [
        f"Ticket ID: {row.ticket_id}\n"
        f"Category: {row.category}\n"
        f"Product: {row.product_name}\n"
        f"Customer message: {row.customer_message}"
        for _, row in complaint_evidence.iterrows()
    ]
)

print(evidence_text[:5000])

Ticket ID: TK-254577
Category: Billing & Payments
Product: Pulse 2 True Wireless Earbuds
Customer message: need GST invoice for my order, i want my money back

Ticket ID: TK-254800
Category: Billing & Payments
Product: Nexa 2 Smartwatch
Customer message: hi sir
i was charged twice for one order
urgent

Ticket ID: TK-254774
Category: Billing & Payments
Product: Pulse True Wireless Earbuds
Customer message: [IVR transcript] Honestly regretting this purchase. I bought the Pulse buds on 19 Jun. double payment deducted. I already waited a week. I want a replacement or refund, nothing else.

Ticket ID: TK-254594
Category: Billing & Payments
Product: Orbit Smart Speaker
Customer message: Got Orbit speaker from vireo.in 2 weeks ago. My company accounts team is asking for the tax bill. I tried different browser. Nothing changed. How do I get this fixed?

Ticket ID: TK-254683
Category: Billing & Payments
Product: Pulse 2 True Wireless Earbuds
Customer message: my bank says Rs 4999 went to you bu

In [179]:
prompt = f"""
You are a customer-support operations analyst for Vireo Audio.

Analyze ONLY the customer messages provided below.

Identify the 3 to 5 most important complaint themes.

For each theme provide:
1. theme_name
2. short_description
3. supporting_ticket_ids

Rules:
- Use only information present in the messages.
- Do not invent causes.
- Do not invent statistics.
- Do not calculate ticket counts.
- Do not claim causation.
- A ticket can belong to only one primary theme.
- If evidence is unclear, say "needs review".

Customer messages:

{evidence_text}
"""

In [181]:
import requests
import time

start = time.time()

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma3:4b",
        "prompt": "In one sentence, what is a customer support ticket?",
        "stream": False
    },
    timeout=60
)

print("Time:", round(time.time() - start, 2), "seconds")
print(response.status_code)
print(response.json()["response"])

Time: 16.99 seconds
200
A customer support ticket is a record of a customer's issue or request, tracked and managed by a support team to ensure it's addressed and resolved.


In [183]:
complaint_evidence_small = (
    current_week_tickets[
        current_week_tickets["category"].isin(
            top_week_categories.index
        )
    ]
    .groupby("category", group_keys=False)
    .apply(
        lambda x: x.sample(
            min(len(x), 3),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print("Evidence rows:", len(complaint_evidence_small))

Evidence rows: 15


C:\Users\pavan\AppData\Local\Temp\ipykernel_20268\767817735.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [184]:
evidence_text = "\n\n".join(
    [
        f"Ticket ID: {row.ticket_id}\n"
        f"Category: {row.category}\n"
        f"Product: {row.product_name}\n"
        f"Customer message: {row.customer_message}"
        for _, row in complaint_evidence_small.iterrows()
    ]
)

In [185]:
prompt = f"""
You are analyzing Vireo Audio customer support tickets.

From the customer messages below:
- identify 3 to 5 recurring complaint themes
- give each theme a short name
- give a one-sentence description
- provide supporting ticket IDs

Use only the supplied messages.
Do not invent causes or statistics.
Do not calculate any metrics.

Messages:

{evidence_text}
"""

In [186]:
start = time.time()

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma3:4b",
        "prompt": prompt,
        "stream": False
    },
    timeout=60
)

print("Time:", round(time.time() - start, 2), "seconds")

response.raise_for_status()

ai_summary = response.json()["response"]

print(ai_summary)

ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=60)

In [187]:
print("Evidence rows:", len(complaint_evidence_small))
print("Evidence characters:", len(evidence_text))
print("Prompt characters:", len(prompt))

Evidence rows: 15
Evidence characters: 3224
Prompt characters: 3570


In [188]:
small_evidence = "\n\n".join(
    [
        f"Ticket: {row.ticket_id}\n"
        f"Category: {row.category}\n"
        f"Complaint: {row.customer_message}"
        for _, row in complaint_evidence_small.head(5).iterrows()
    ]
)

small_prompt = f"""
Identify the main complaint themes in these Vireo support tickets.
Return only 3 bullet points.
Use only the evidence provided.

{small_evidence}
"""

In [189]:
start = time.time()

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma3:4b",
        "prompt": small_prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 150
        }
    },
    timeout=60
)

print("Time:", round(time.time() - start, 2), "seconds")

response.raise_for_status()
print(response.json()["response"])

Time: 33.35 seconds
Here are the main complaint themes based on the provided Vireo support tickets:

*   **Duplicate Billing:** Multiple tickets (TK-254800, TK-254774) report being charged more than once for an order.
*   **Invoice Requests:** One ticket (TK-254577) specifically requests a GST invoice.
*   **Charging Issues:** A ticket (TK-254660) details a product (Pulse) that isn't charging correctly.


In [190]:
full_evidence_text = "\n\n".join(
    [
        f"Ticket ID: {row.ticket_id}\n"
        f"Category: {row.category}\n"
        f"Product: {row.product_name}\n"
        f"Customer message: {row.customer_message}"
        for _, row in complaint_evidence_small.iterrows()
    ]
)

print("Characters:", len(full_evidence_text))

Characters: 3224


In [191]:
full_prompt = f"""
You are a customer-support operations analyst for Vireo Audio.

From the customer messages below, identify the 3 most important recurring complaint themes.

For each theme provide exactly:
1. Theme name
2. One short description
3. Two supporting ticket IDs

Rules:
- Use only the supplied messages.
- Do not invent causes.
- Do not invent statistics.
- Do not calculate metrics.
- Keep every theme concise.
- Return ONLY the 3 themes.
- Do not add an introduction or conclusion.

Customer messages:

{full_evidence_text}
"""

In [192]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma3:4b",
        "prompt": full_prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 250
        }
    },
    timeout=90
)

response.raise_for_status()

ai_summary = response.json()["response"]

print(ai_summary)

1.  Theme name
    Double Billing
    Description
    Customers are being charged more than once for their orders.
    Supporting ticket IDs
    TK-254800
    TK-254774

2.  Theme name
    Refund Delays
    Description
    Customers are experiencing significant delays in receiving promised refunds.
    Supporting ticket IDs
    TK-254721
    TK-254632

3.  Theme name
    Delivery Issues
    Description
    Customers are reporting issues with their deliveries, including incorrect items and order status discrepancies.
    Supporting ticket IDs
    TK-254789
    TK-254684


In [193]:
with open("../data/latest_ai_summary.txt", "w", encoding="utf-8") as f:
    f.write(ai_summary)

print("Saved.")

Saved.


In [194]:
def format_digest(digest, ai_summary=None):

    text = f"""
WEEKLY SUPPORT DIGEST
Week: {digest["week"]}

WHAT CHANGED
Support handled {digest["tickets"]} tickets,
{digest["ticket_change_pct"]:+.1f}% versus the previous week.

REPEAT CONTACT
{digest["repeat_candidates"]} tickets were repeat-contact candidates
({digest["repeat_rate_pct"]:.1f}%).
Estimated candidate repeat-contact cost: ₹{digest["repeat_contact_cost"]:,}.

SLA
{digest["sla_breaches"]} tickets breached first-response SLA
({digest["sla_breach_rate_pct"]:.1f}%).
Estimated SLA credit exposure: ₹{digest["sla_credit_exposure"]:,}.

CUSTOMER EXPERIENCE
Average CSAT: {digest["avg_csat"]:.2f}/5.

OPERATIONS
Transfers: {digest["transfers"]}

TOP CATEGORIES
"""

    for item in digest["top_categories"]:
        text += (
            f'- {item["category"]}: '
            f'{item["tickets"]} tickets\n'
        )

    text += "\nTOP REPEAT-CONTACT CATEGORIES\n"

    for item in digest["top_repeat_categories"]:
        text += (
            f'- {item["category"]}: '
            f'{item["repeat_candidates"]} repeat candidates\n'
        )

    if ai_summary:
        text += "\nAI COMPLAINT SUMMARY\n"
        text += ai_summary

    return text

In [195]:
final_digest = format_digest(
    digest,
    ai_summary=ai_summary
)

print(final_digest)


WEEKLY SUPPORT DIGEST
Week: 22 Jun 2026

WHAT CHANGED
Support handled 199 tickets,
+19.2% versus the previous week.

REPEAT CONTACT
23 tickets were repeat-contact candidates
(11.6%).
Estimated candidate repeat-contact cost: ₹6,150.

SLA
15 tickets breached first-response SLA
(7.5%).
Estimated SLA credit exposure: ₹5,250.

CUSTOMER EXPERIENCE
Average CSAT: 3.31/5.

OPERATIONS
Transfers: 21

TOP CATEGORIES
- Delivery & Shipping: 34 tickets
- Charging & Battery: 28 tickets
- Other: 28 tickets
- Returns & Refunds: 23 tickets
- Billing & Payments: 20 tickets

TOP REPEAT-CONTACT CATEGORIES
- Other: 7 repeat candidates
- Delivery & Shipping: 4 repeat candidates
- Returns & Refunds: 3 repeat candidates
- App & Firmware: 2 repeat candidates
- Audio Quality: 2 repeat candidates

AI COMPLAINT SUMMARY
1.  Theme name
    Double Billing
    Description
    Customers are being charged more than once for their orders.
    Supporting ticket IDs
    TK-254800
    TK-254774

2.  Theme name
    Refund De

In [196]:
agents = pd.read_csv("../data/agents.csv")

agents["from_date"] = pd.to_datetime(agents["from_date"])
agents["to_date"] = pd.to_datetime(agents["to_date"])
agents.head()


,agent_id,name,site,team,shift,tier,from_date,to_date
0,A3001,Shreya Kumar,Indore,Chat Frontline,Night,1,2023-12-29,NaT
1,A3002,Zoya Srivastava,Indore,Chat Frontline,Night,1,2022-10-25,NaT
2,A3003,Rohit Yadav,Indore,Chat Frontline,Night,1,2023-03-24,NaT
3,A3004,Aishwarya Shinde,Bengaluru,Chat Frontline,Morning,1,2021-04-25,NaT
4,A3005,Sameer Menon,Indore,Chat Frontline,Day,1,2022-06-10,NaT


In [197]:
print("Agent rows:", len(agents))
print("Unique agents:", agents["agent_id"].nunique())

Agent rows: 44
Unique agents: 44


In [198]:
leaderboard_df = final_tickets.merge(
    agents[
        [
            "agent_id",
            "name",
            "team",
            "tier",
            "site",
            "shift",
            "from_date",
            "to_date"
        ]
    ],
    on="agent_id",
    how="left"
)

In [199]:
print(
    "Tickets without agent match:",
    leaderboard_df["name"].isna().sum()
)

Tickets without agent match: 0


In [200]:
week_end = latest_complete_week + pd.Timedelta(days=6)

print("Leaderboard week:",
      latest_complete_week.date(),
      "to",
      week_end.date())

Leaderboard week: 2026-06-22 to 2026-06-28


In [201]:
weekly_completed = leaderboard_df[
    (leaderboard_df["week"] == latest_complete_week) &
    (leaderboard_df["status"].isin(["resolved", "closed"]))
].copy()

In [202]:
tier1_weekly = weekly_completed[
    weekly_completed["tier"] == 1
].copy()

In [203]:
leaderboard = (
    tier1_weekly
    .groupby(
        ["agent_id", "name", "team", "site", "shift"],
        as_index=False
    )
    .agg(
        tickets_completed=("ticket_id", "count"),
        sla_breaches=("sla_breach", "sum"),
        avg_csat=("csat_score", "mean"),
        transfers=("transfers", "sum")
    )
)

In [204]:
leaderboard["sla_breach_rate"] = (
    leaderboard["sla_breaches"]
    / leaderboard["tickets_completed"]
)

In [205]:
leaderboard = leaderboard.sort_values(
    "tickets_completed",
    ascending=False
)

leaderboard

,agent_id,name,team,site,shift,tickets_completed,sla_breaches,avg_csat,transfers,sla_breach_rate
36,A3037,Vivaan Sethi,Returns Desk,Bengaluru,Day,12,1,3.428571,0,0.083333
20,A3021,Pooja Dhillon,Email Frontline,Bengaluru,Day,11,1,3.0,1,0.090909
32,A3033,Diya Singh,Billing,Bengaluru,Day,10,0,3.6,0,0.000000
28,A3029,Geeta Rathore,Logistics,Indore,Day,10,0,2.5,0,0.000000
9,A3010,Om Varghese,Chat Frontline,Indore,Day,9,0,3.5,0,0.000000
30,A3031,Aishwarya Agarwal,Logistics,Indore,Day,8,1,3.25,0,0.125000
18,A3019,Kavya D'Souza,Email Frontline,Bengaluru,Day,8,0,4.333333,1,0.000000
15,A3016,Ayaan Pawar,Email Frontline,Indore,Night,7,1,3.0,0,0.142857
35,A3036,Saanvi Saxena,Returns Desk,Bengaluru,Morning,7,3,3.0,2,0.428571
26,A3027,Rajat Saxena,Logistics,Indore,Morning,7,0,3.666667,2,0.000000


In [206]:
leaderboard = leaderboard.sort_values(
    "tickets_completed",
    ascending=False
)

leaderboard

,agent_id,name,team,site,shift,tickets_completed,sla_breaches,avg_csat,transfers,sla_breach_rate
36,A3037,Vivaan Sethi,Returns Desk,Bengaluru,Day,12,1,3.428571,0,0.083333
20,A3021,Pooja Dhillon,Email Frontline,Bengaluru,Day,11,1,3.0,1,0.090909
32,A3033,Diya Singh,Billing,Bengaluru,Day,10,0,3.6,0,0.000000
28,A3029,Geeta Rathore,Logistics,Indore,Day,10,0,2.5,0,0.000000
9,A3010,Om Varghese,Chat Frontline,Indore,Day,9,0,3.5,0,0.000000
30,A3031,Aishwarya Agarwal,Logistics,Indore,Day,8,1,3.25,0,0.125000
18,A3019,Kavya D'Souza,Email Frontline,Bengaluru,Day,8,0,4.333333,1,0.000000
15,A3016,Ayaan Pawar,Email Frontline,Indore,Night,7,1,3.0,0,0.142857
35,A3036,Saanvi Saxena,Returns Desk,Bengaluru,Morning,7,3,3.0,2,0.428571
26,A3027,Rajat Saxena,Logistics,Indore,Morning,7,0,3.666667,2,0.000000


In [207]:
print("Tickets before agent merge:", len(weekly_completed))
print("Tickets after agent merge:", len(leaderboard_df))

Tickets before agent merge: 186
Tickets after agent merge: 11875


In [208]:
agent_assignment_counts = (
    agents.groupby("agent_id")
    .size()
    .sort_values(ascending=False)
)

print(agent_assignment_counts.head(10))

agent_id
A3001    1
A3002    1
A3025    1
A3026    1
A3027    1
A3028    1
A3029    1
A3030    1
A3031    1
A3032    1
dtype: int64


In [209]:
print(
    "Agents with multiple roster rows:",
    (agent_assignment_counts > 1).sum()
)

Agents with multiple roster rows: 0


In [210]:
leaderboard_base = weekly_completed.copy()

leaderboard_base = leaderboard_base.merge(
    agents,
    on="agent_id",
    how="left"
)

leaderboard_base["assignment_match"] = (
    (leaderboard_base["resolved_at_normalized"] >= leaderboard_base["from_date"]) &
    (
        leaderboard_base["to_date"].isna() |
        (leaderboard_base["resolved_at_normalized"] <= leaderboard_base["to_date"])
    )
)

leaderboard_base = leaderboard_base[
    leaderboard_base["assignment_match"]
].copy()

KeyError: 'from_date'

In [211]:
print(agents.columns.tolist())

['agent_id', 'name', 'site', 'team', 'shift', 'tier', 'from_date', 'to_date']


In [212]:
print(agents.head())


  agent_id              name       site            team    shift  tier  \
0    A3001      Shreya Kumar     Indore  Chat Frontline    Night     1   
1    A3002   Zoya Srivastava     Indore  Chat Frontline    Night     1   
2    A3003       Rohit Yadav     Indore  Chat Frontline    Night     1   
3    A3004  Aishwarya Shinde  Bengaluru  Chat Frontline  Morning     1   
4    A3005      Sameer Menon     Indore  Chat Frontline      Day     1   

   from_date to_date  
0 2023-12-29     NaT  
1 2022-10-25     NaT  
2 2023-03-24     NaT  
3 2021-04-25     NaT  
4 2022-06-10     NaT  


In [213]:
# 1. Start fresh from the ticket data
weekly_completed_raw = final_tickets[
    (final_tickets["week"] == latest_complete_week) &
    (final_tickets["status"].isin(["resolved", "closed"]))
].copy()

print("Completed tickets before roster merge:", len(weekly_completed_raw))

Completed tickets before roster merge: 186


In [214]:
agents["from_date"] = pd.to_datetime(agents["from_date"])
agents["to_date"] = pd.to_datetime(agents["to_date"])

In [215]:
leaderboard_base = weekly_completed_raw.merge(
    agents[
        [
            "agent_id",
            "name",
            "team",
            "tier",
            "site",
            "shift",
            "from_date",
            "to_date"
        ]
    ],
    on="agent_id",
    how="left"
)

In [216]:
leaderboard_base["assignment_match"] = (
    (leaderboard_base["resolved_at_normalized"] >= leaderboard_base["from_date"]) &
    (
        leaderboard_base["to_date"].isna() |
        (leaderboard_base["resolved_at_normalized"] <= leaderboard_base["to_date"])
    )
)

In [217]:
leaderboard_base = leaderboard_base[
    leaderboard_base["assignment_match"]
].copy()

In [218]:
print("Completed tickets:", len(weekly_completed_raw))
print("After roster matching:", len(leaderboard_base))

Completed tickets: 186
After roster matching: 186


In [219]:
tier1_weekly = leaderboard_base[
    leaderboard_base["tier"] == 1
].copy()


In [220]:
print("Completed tickets before roster merge:", len(weekly_completed_raw))
print("After roster matching:", len(leaderboard_base))

Completed tickets before roster merge: 186
After roster matching: 186


In [221]:
tier1_weekly = leaderboard_base[
    leaderboard_base["tier"] == 1
].copy()

leaderboard = (
    tier1_weekly
    .groupby(
        ["agent_id", "name", "team", "site", "shift"],
        as_index=False
    )
    .agg(
        tickets_completed=("ticket_id", "count"),
        sla_breaches=("sla_breach", "sum"),
        avg_csat=("csat_score", "mean"),
        transfers=("current_helpdesk_transfer", "sum")
    )
)

leaderboard["sla_breach_rate"] = (
    leaderboard["sla_breaches"]
    / leaderboard["tickets_completed"]
)

leaderboard = leaderboard.sort_values(
    "tickets_completed",
    ascending=False
)

leaderboard

,agent_id,name,team,site,shift,tickets_completed,sla_breaches,avg_csat,transfers,sla_breach_rate
36,A3037,Vivaan Sethi,Returns Desk,Bengaluru,Day,12,1,3.428571,0,0.083333
20,A3021,Pooja Dhillon,Email Frontline,Bengaluru,Day,11,1,3.0,1,0.090909
32,A3033,Diya Singh,Billing,Bengaluru,Day,10,0,3.6,0,0.000000
28,A3029,Geeta Rathore,Logistics,Indore,Day,10,0,2.5,0,0.000000
9,A3010,Om Varghese,Chat Frontline,Indore,Day,9,0,3.5,0,0.000000
30,A3031,Aishwarya Agarwal,Logistics,Indore,Day,8,1,3.25,0,0.125000
18,A3019,Kavya D'Souza,Email Frontline,Bengaluru,Day,8,0,4.333333,1,0.000000
15,A3016,Ayaan Pawar,Email Frontline,Indore,Night,7,1,3.0,0,0.142857
35,A3036,Saanvi Saxena,Returns Desk,Bengaluru,Morning,7,3,3.0,2,0.428571
26,A3027,Rajat Saxena,Logistics,Indore,Morning,7,0,3.666667,2,0.000000


In [222]:
print(
    "Tier 2 rows in leaderboard:",
    len(
        leaderboard[
            leaderboard["team"] == "Escalations & Warranty"
        ]
    )
)

Tier 2 rows in leaderboard: 0


In [223]:
print("Tier 1 completed tickets:", len(tier1_weekly))
print("Leaderboard total:", leaderboard["tickets_completed"].sum())

print(
    "Unmatched agent names:",
    leaderboard_base["name"].isna().sum()
)

Tier 1 completed tickets: 177
Leaderboard total: 177
Unmatched agent names: 0


In [2]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent / "src"))

from metrics import (
    add_week_column,
    calculate_weekly_kpis,
    calculate_repeat_cost
)

In [225]:
test_df = add_week_column(final_tickets)

print(test_df[["created_at", "week"]].head())

           created_at       week
0 2025-01-01 09:48:00 2024-12-30
1 2025-01-01 13:24:00 2024-12-30
2 2025-01-01 13:55:00 2024-12-30
3 2025-01-01 15:13:00 2024-12-30
4 2025-01-01 20:24:00 2024-12-30


In [226]:
test_weekly = calculate_weekly_kpis(final_tickets)

print(test_weekly.tail())

            tickets  repeat_candidates  sla_breaches  avg_csat  transfers  \
week                                                                        
2026-06-01      171                 26            15  3.186441         12   
2026-06-08      203                 22            21  3.202247         20   
2026-06-15      167                 19            19  3.152778         21   
2026-06-22      199                 23            15    3.3125         21   
2026-06-29       46                  6             2  3.217391          8   

            refunds  refund_amount  repeat_rate  sla_breach_rate  \
week                                                               
2026-06-01       26        73729.0     0.152047         0.087719   
2026-06-08       30        77355.0     0.108374         0.103448   
2026-06-15       39       119834.0     0.113772         0.113772   
2026-06-22       30       125797.0     0.115578         0.075377   
2026-06-29       12        33085.0     0.130435     

In [227]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent / "src"))

from cleaning import load_tickets, clean_tickets

In [228]:
raw_tickets = load_tickets("../data/tickets.csv")

test_clean = clean_tickets(raw_tickets)

print("Raw rows:", len(raw_tickets))
print("Clean rows:", len(test_clean))
print("Unique ticket IDs:", test_clean["ticket_id"].nunique())
print(
    "Duplicate IDs:",
    test_clean["ticket_id"].duplicated().sum()
)

Raw rows: 12528
Clean rows: 11875
Unique ticket IDs: 11875
Duplicate IDs: 0


In [229]:
print(
    "Resolution before creation:",
    (
        test_clean["resolved_at_normalized"]
        < test_clean["created_at"]
    ).sum()
)

Resolution before creation: 0


In [230]:
print(
    test_clean[
        [
            "ticket_id",
            "source_system",
            "resolved_at",
            "resolved_at_normalized"
        ]
    ].head(10)
)

   ticket_id source_system         resolved_at resolved_at_normalized
0  TK-240001      helpdesk 2025-01-01 11:49:00    2025-01-01 11:49:00
1  TK-240002     legacy_fd 2025-01-01 08:28:00    2025-01-01 13:58:00
2  TK-240003     legacy_fd 2025-01-01 09:07:00    2025-01-01 14:37:00
3  TK-240004     legacy_fd 2025-01-02 11:57:00    2025-01-02 17:27:00
4  TK-240005      helpdesk 2025-01-03 20:52:00    2025-01-03 20:52:00
5  TK-240006     legacy_fd 2025-01-01 17:43:00    2025-01-01 23:13:00
6  TK-240007     legacy_fd 2025-01-01 21:08:00    2025-01-02 02:38:00
7  TK-240008     legacy_fd 2025-01-07 03:08:00    2025-01-07 08:38:00
8  TK-240009     legacy_fd 2025-01-07 05:58:00    2025-01-07 11:28:00
9  TK-240010     legacy_fd 2025-01-02 12:17:00    2025-01-02 17:47:00


In [231]:
from repeat_contacts import find_repeat_candidates

In [232]:
repeat_pairs_test, repeat_tickets_test = (
    find_repeat_candidates(final_tickets)
)


In [233]:
print(
    "Candidate repeat pairs:",
    len(repeat_pairs_test)
)

print(
    "Unique candidate repeat tickets:",
    len(repeat_tickets_test)
)

Candidate repeat pairs: 1527
Unique candidate repeat tickets: 1414


In [234]:
print(
    "Candidate repeat-contact cost: ₹",
    repeat_tickets_test["repeat_contact_cost"].sum()
)

Candidate repeat-contact cost: ₹ 367500


In [235]:
repeat_tickets_test[
    [
        "new_ticket_id",
        "previous_ticket_id",
        "customer_id",
        "product_sku",
        "category",
        "days_after_resolution",
        "repeat_contact_cost",
    ]
].head(20)

,new_ticket_id,previous_ticket_id,customer_id,product_sku,category,days_after_resolution,repeat_contact_cost
349,TK-243900,TK-243899,C100833,VA-HP-ST3,Billing & Payments,0.020139,210
61,TK-240929,TK-240926,C102281,VA-SW-FIT,Billing & Payments,0.059722,260
617,TK-246482,TK-246481,C108097,VA-EB-PL1,Billing & Payments,0.063194,210
985,TK-249729,TK-249726,C101465,VA-AC-CBL,Other,0.075694,210
1277,TK-252527,TK-252517,C103720,VA-EB-PL2,Other,0.186806,210
0,TK-240080,TK-240065,C106846,VA-AC-CH65,Delivery & Shipping,0.248611,520
1456,TK-254177,TK-254160,C106434,VA-AC-CH65,Billing & Payments,0.300694,520
3,TK-240221,TK-240216,C108634,VA-EB-AIR,Billing & Payments,0.348611,520
1068,TK-250597,TK-250586,C102761,VA-SP-MINI,Other,0.396528,210
64,TK-240968,TK-240966,C104993,VA-HP-ST2,Other,0.550694,260


In [236]:
from digest import (
    build_weekly_digest,
    format_weekly_digest
)

In [237]:
test_digest = build_weekly_digest(
    weekly_kpis=weekly_kpis,
    weekly_category_counts=weekly_category_counts,
    final_tickets=final_tickets,
    week_start=latest_complete_week,
)

In [238]:
print(test_digest)

{'week': '22 Jun 2026', 'tickets': 199, 'previous_tickets': 167, 'ticket_change_pct': 19.16, 'repeat_candidates': 23, 'repeat_rate_pct': 11.56, 'repeat_contact_cost': 6150, 'sla_breaches': 15, 'sla_breach_rate_pct': 7.54, 'sla_credit_exposure': 5250, 'avg_csat': 3.31, 'transfers': 21, 'refunds': 30, 'refund_amount': 125797, 'top_categories': [{'category': 'Delivery & Shipping', 'tickets': 34}, {'category': 'Charging & Battery', 'tickets': 28}, {'category': 'Other', 'tickets': 28}, {'category': 'Returns & Refunds', 'tickets': 23}, {'category': 'Billing & Payments', 'tickets': 20}], 'top_repeat_categories': [{'category': 'Other', 'repeat_candidates': 7}, {'category': 'Delivery & Shipping', 'repeat_candidates': 4}, {'category': 'Returns & Refunds', 'repeat_candidates': 3}, {'category': 'App & Firmware', 'repeat_candidates': 2}, {'category': 'Audio Quality', 'repeat_candidates': 2}]}


In [239]:
print(
    format_weekly_digest(
        test_digest,
        ai_summary=ai_summary
    )
)

WEEKLY SUPPORT DIGEST
Week: 22 Jun 2026

WHAT CHANGED
Support handled 199 tickets,
+19.2% versus the previous week.

REPEAT CONTACT
23 tickets were repeat-contact candidates
(11.6%).
Estimated candidate repeat-contact cost:
₹6,150.

SLA
15 tickets breached first-response SLA
(7.5%).
Estimated SLA credit exposure:
₹5,250.

CUSTOMER EXPERIENCE
Average CSAT: 3.31/5.

OPERATIONS
Transfers: 21

TOP CATEGORIES
- Delivery & Shipping: 34 tickets
- Charging & Battery: 28 tickets
- Other: 28 tickets
- Returns & Refunds: 23 tickets
- Billing & Payments: 20 tickets

TOP REPEAT-CONTACT CATEGORIES
- Other: 7 repeat candidates
- Delivery & Shipping: 4 repeat candidates
- Returns & Refunds: 3 repeat candidates
- App & Firmware: 2 repeat candidates
- Audio Quality: 2 repeat candidates

AI COMPLAINT SUMMARY
1.  Theme name
    Double Billing
    Description
    Customers are being charged more than once for their orders.
    Supporting ticket IDs
    TK-254800
    TK-254774

2.  Theme name
    Refund Del

In [240]:
from ai_summary import (
    build_complaint_evidence,
    generate_complaint_summary,
)

In [241]:
weekly_evidence = build_complaint_evidence(
    current_week_tickets
)

print("Evidence rows:", len(weekly_evidence))

Evidence rows: 15


In [242]:
ai_summary_test, generation_time = generate_complaint_summary(
    weekly_evidence
)

print("Generation time:", round(generation_time, 2), "seconds")
print(ai_summary_test)

Generation time: 65.95 seconds
1.  Theme name: Duplicate Billing
    Description: Customers are being charged incorrectly for orders.
    Supporting ticket IDs: TK-254800, TK-254774

2.  Theme name: Refund Delays
    Description: Promised refunds are not being processed within the agreed timeframe.
    Supporting ticket IDs: TK-254721, TK-254632

3.  Theme name: Order Delivery Issues
    Description: Customers are experiencing problems with order delivery or order discrepancies.
    Supporting ticket IDs: TK-254684, TK-254789


In [243]:
from leaderboard import build_tier1_leaderboard

In [244]:
test_leaderboard = build_tier1_leaderboard(
    weekly_completed,
    agents,
)

print(test_leaderboard)

   agent_id                name             team       site    shift  \
0     A3037        Vivaan Sethi     Returns Desk  Bengaluru      Day   
1     A3021       Pooja Dhillon  Email Frontline  Bengaluru      Day   
2     A3033          Diya Singh          Billing  Bengaluru      Day   
3     A3029       Geeta Rathore        Logistics     Indore      Day   
4     A3010         Om Varghese   Chat Frontline     Indore      Day   
5     A3031   Aishwarya Agarwal        Logistics     Indore      Day   
6     A3019       Kavya D'Souza  Email Frontline  Bengaluru      Day   
7     A3016         Ayaan Pawar  Email Frontline     Indore    Night   
8     A3036       Saanvi Saxena     Returns Desk  Bengaluru  Morning   
9     A3027        Rajat Saxena        Logistics     Indore  Morning   
10    A3038         Pooja Patil     Returns Desk     Indore  Morning   
11    A3034  Manpreet Mukherjee          Billing     Indore  Morning   
12    A3028       Tarun Pereira        Logistics  Bengaluru  Mor

In [11]:
from pipeline import build_analysis_dataset

In [12]:
app_tickets, app_repeat_pairs, app_repeat_tickets = (
    build_analysis_dataset(
        "../data/tickets.csv",
        "../data/products.csv",
    )
)

In [13]:
print("Rows:", len(app_tickets))
print("Unique tickets:", app_tickets["ticket_id"].nunique())

print(
    "Repeat pairs:",
    len(app_repeat_pairs)
)

print(
    "Unique repeat candidates:",
    len(app_repeat_tickets)
)

print(
    "SLA breaches:",
    app_tickets["sla_breach"].sum()
)

Rows: 11875
Unique tickets: 11875
Repeat pairs: 1527
Unique repeat candidates: 1414
SLA breaches: 1051


In [14]:
from metrics import calculate_weekly_kpis

test_weekly = calculate_weekly_kpis(app_tickets)

print(
    test_weekly.loc[
        pd.Timestamp("2026-06-22"),
        ["tickets", "repeat_candidates", "repeat_contact_cost",
         "sla_breaches", "avg_csat"]
    ]
)

tickets                 199.0000
repeat_candidates        23.0000
repeat_contact_cost    6150.0000
sla_breaches             15.0000
avg_csat                  3.3125
Name: 2026-06-22 00:00:00, dtype: float64
